<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_kvasir_capsule_data_lake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phase 2 - Build the raw medical data lake

This is the full notebook aggregating all the code for data lake. Also called data-engineering and data-curation phase as we build a structured raw medical data lake from capsule-endoscopy data before creating visual artifacts or question-answer pairs. The primary dataset is Kvasir-Capsule, which contains capsule-endoscopy videos, labelled images, medically verified finding classes, bounding-box annotations, video identifiers, and frame numbers. The purpose of this phase is to transform the original dataset files into a clean, searchable, and reproducible project data layer.

### 1. Install dependencies and imports

In [ ]:
%pip install -q \
    pandas \
    numpy \
    pyarrow \
    opencv-python-headless \
    scikit-image \
    scikit-learn \
    iterative-stratification \
    tqdm \
    osfclient

In [ ]:
from pathlib import Path
from collections import defaultdict

import re
import json
import random
import warnings
import subprocess
import shutil
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import cv2
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from skimage.metrics import structural_similarity as ssim

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

### 2. Load declarative configuration and reproducibility

In [ ]:
TIMEZONE = "America/Toronto"

RUN_ID = datetime.now(
    ZoneInfo(TIMEZONE)
).strftime(
    "%Y%m%d_%H%M%S"
)


CONFIG = {
    # --------------------------------------------------------------
    # Experiment identity
    # --------------------------------------------------------------

    "phase": "phase2",

    "experiment_name": (
        "phase2_kvasir_capsule_"
        "raw_medical_data_lake"
    ),

    "notebook_name": (
        "phase2_kvasir_capsule_"
        "data_lake.ipynb"
    ),

    "seed": 42,
    "run_id": RUN_ID,
    "timezone": TIMEZONE,

    # --------------------------------------------------------------
    # Storage
    # --------------------------------------------------------------

    "storage_backend": "google_drive",

    "storage_root": (
        "/content/drive/MyDrive/"
        "MMVQA_Clinical"
    ),

    "raw_data_dir": (
        "data/raw/kvasir_capsule"
    ),

    "interim_data_dir": (
        "data/interim/phase2"
    ),

    "curated_data_dir": (
        "data/curated/phase2"
    ),

    "output_dir": (
        "outputs/phase2"
    ),

    # --------------------------------------------------------------
    # Dataset source
    # --------------------------------------------------------------

    "dataset_name": "Kvasir-Capsule",
    "dataset_source": "OSF",
    "dataset_osf_project_id": "dv2ag",

    "dataset_download_enabled": False,

    # --------------------------------------------------------------
    # Raw dataset availability validation
    # --------------------------------------------------------------

    "dataset_validation": {
        "required_metadata_file": ("metadata.csv"),
        "osf_storage_subdir": "osfstorage",

        # Official Kvasir-Capsule labelled-image directory.
        "labelled_images_subdir": "images",

        "minimum_video_files": 117,

        "minimum_labelled_images": 47238,

        "image_extensions": [
            ".png",
            ".jpg",
            ".jpeg",
        ],

        "video_extensions": [
            ".avi",
            ".mp4",
            ".mkv",
        ],
    },

    # --------------------------------------------------------------
    # Expected dataset characteristics
    # --------------------------------------------------------------

    "expected_labelled_frames": 47238,
    "expected_classes": 14,

    "expected_labelled_videos": 43,

    "expected_unlabelled_videos": 74,

    "expected_total_videos": 117,

    "expected_total_extractable_frames": (
        4741504
    ),

    "expected_unlabelled_frames": (
        4694266
    ),

    # --------------------------------------------------------------
    # Video and frame alignment
    # --------------------------------------------------------------


    "expected_export_container_fps": 30.0,

    "frame_index_offset_candidates": [
        -1,
        0,
        1,
    ],

    "frame_alignment_sample_size": 30,
    "frame_alignment_min_valid_fraction": 0.80,

    "frame_alignment_comparison_size": [
        128,
        128,
    ],

    # --------------------------------------------------------------
    # Video-level split
    # --------------------------------------------------------------

    "split_strategy": (
        "video_level_multilabel_stratified"
    ),

    "train_fraction": 0.70,
    "validation_fraction": 0.15,
    "test_fraction": 0.15,

    # --------------------------------------------------------------
    # Verified finding segments
    # --------------------------------------------------------------

    "segment_max_gap_frames": 1,

    "minimum_verified_segment_frames": 2,

    # --------------------------------------------------------------
    # Domain adaptation
    # --------------------------------------------------------------

    "domain_adaptation_enabled": True,

    "domain_adaptation_include_fully_unlabelled_videos": (
        True
    ),

    "domain_adaptation_include_train_video_unlabelled_frames": (
        True
    ),

    "domain_adaptation_sampling_seconds": 0.5,


    # --------------------------------------------------------------
    # Temporal context
    # --------------------------------------------------------------

    "temporal_context_enabled": True,

    "temporal_context_offsets_seconds": [
        -2.0,
        -1.0,
        -0.5,
        0.0,
        0.5,
        1.0,
        2.0,
    ],

    "temporal_context_extract_frames": True,

    "temporal_context_image_format": "jpg",

    "temporal_context_jpeg_quality": 95,

    # --------------------------------------------------------------
    # Image quality control
    # --------------------------------------------------------------

    # Pixel-level measurement parameters

    "qc_underexposed_pixel_threshold": 10,

    "qc_overexposed_pixel_threshold": 245,

    "qc_specular_value_threshold": 245,

    "qc_specular_saturation_threshold": 40,

    "qc_fov_radius_fraction": 0.95,

    "qc_enabled": True,

    "qc_blur_quantile": 0.05,

    "qc_contrast_quantile": 0.05,

    "qc_brightness_low_quantile": 0.01,

    "qc_brightness_high_quantile": 0.99,

    # --------------------------------------------------------------
    # Normalized clinical taxonomy
    # --------------------------------------------------------------

    # Keys must match finding_class_normalized.
    "clinical_group_map": {
        "ampulla of vater": (
            "anatomical_landmark"
        ),

        "ileocecal valve": (
            "anatomical_landmark"
        ),

        "pylorus": (
            "anatomical_landmark"
        ),

        "normal clean mucosa": (
            "normal_mucosa"
        ),

        "reduced mucosal view": (
            "visibility_limitation"
        ),

        "blood fresh": "bleeding",

        "blood hematin": "bleeding",

        "angiectasia": (
            "vascular_lesion"
        ),

        "erosion": (
            "mucosal_lesion"
        ),

        "erythema": (
            "mucosal_lesion"
        ),

        "ulcer": (
            "mucosal_lesion"
        ),

        "lymphangiectasia": (
            "lymphatic_lesion"
        ),

        "polyp": (
            "protruding_lesion"
        ),

        "foreign body": (
            "foreign_body"
        ),
    },
}


CONFIG

In [ ]:

# Set random seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"Seed set to: {CONFIG['seed']}")

### 3. Mount Google Drive Storage Backend

In [ ]:
def mount_storage(config):
    """
    Mounts persistent storage when required by the configured backend.

    For Google Colab + Google Drive, this mounts Drive under /content/drive.
    It does not download or copy the dataset.
    """

    storage_backend = config["storage_backend"]

    if storage_backend == "google_drive":

        try:
            from google.colab import drive

            drive.mount(
                "/content/drive",
                force_remount=False,
            )

            print("Google Drive mounted.")

        except ImportError:
            raise RuntimeError(
                "Google Drive backend is configured, "
                "but the notebook is not running in Google Colab."
            )

    elif storage_backend == "local":

        print("Using local storage.")

    else:

        raise ValueError(
            f"Unsupported storage backend: {storage_backend}"
        )


mount_storage(CONFIG)

### 4. Define data paths

In [ ]:
def prepare_phase2_dirs(config):
    """
    Resolves and creates the directory structure required
    for Phase 2.

    Main directories are declared in CONFIG. Relative paths
    are resolved against storage_root, while absolute paths
    are preserved.

    Raw-dataset reference paths are returned, but dataset
    source files and source subdirectories are not created.

    Side effect:
        Creates writable Phase 2 directories on persistent
        storage.

    Returns:
        Dictionary containing resolved Path objects.
    """

    storage_root = Path(
        config["storage_root"]
    )

    validation = config[
        "dataset_validation"
    ]


    def resolve_path(path_value):
        """
        Resolves a CONFIG path against storage_root.

        Absolute paths are returned unchanged.
        """

        path = Path(path_value)

        if path.is_absolute():
            return path

        return storage_root / path


    # --------------------------------------------------------------
    # Main CONFIG directories
    # --------------------------------------------------------------

    raw_data_dir = resolve_path(
        config["raw_data_dir"]
    )

    interim_data_dir = resolve_path(
        config["interim_data_dir"]
    )

    curated_data_dir = resolve_path(
        config["curated_data_dir"]
    )

    output_dir = resolve_path(
        config["output_dir"]
    )


    # --------------------------------------------------------------
    # Raw-dataset source references
    # --------------------------------------------------------------

    dataset_root_dir = (
        raw_data_dir
        / validation["osf_storage_subdir"]
    )

    labelled_images_dir = (
        dataset_root_dir
        / validation["labelled_images_subdir"]
    )

    metadata_path = (
        dataset_root_dir
        / validation["required_metadata_file"]
    )


    # --------------------------------------------------------------
    # Writable directories created by the pipeline
    # --------------------------------------------------------------

    created_directories = {
        # Main directories
        "raw_data_dir":
            raw_data_dir,

        "interim_data_dir":
            interim_data_dir,

        "curated_data_dir":
            curated_data_dir,

        "output_dir":
            output_dir,

        # Derived data directories
        "temporal_frames_dir":
            interim_data_dir
            / "temporal_frames",

        "manifests_dir":
            curated_data_dir
            / "manifests",

        "splits_dir":
            curated_data_dir
            / "splits",

        # Derived output directories
        "configs_dir":
            output_dir
            / "configs",

        "results_dir":
            output_dir
            / "results",

        "reports_dir":
            output_dir
            / "reports",
    }


    # --------------------------------------------------------------
    # Create only writable pipeline directories
    # --------------------------------------------------------------

    for directory in created_directories.values():
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )


    # --------------------------------------------------------------
    # Dataset paths that must come from the OSF dataset
    # --------------------------------------------------------------

    dataset_paths = {
        "dataset_root_dir":
            dataset_root_dir,

        "labelled_images_dir":
            labelled_images_dir,

        "metadata_path":
            metadata_path,
    }


    return {
        **created_directories,
        **dataset_paths,
    }


DIRS = prepare_phase2_dirs(
    CONFIG
)

DIRS

### 5. Dataset verification

In [ ]:
def build_raw_dataset_inventory(
    config,
    dirs,
):
    """
    Builds a declarative inventory of the raw
    Kvasir-Capsule dataset.

    The filesystem is inspected but not modified.

    Returns:
        One dictionary containing paths, observed counts,
        expected minimums, and the final validation result.
    """

    required_dir_keys = {
        "raw_data_dir",
        "dataset_root_dir",
        "labelled_images_dir",
        "metadata_path",
    }

    missing_dir_keys = sorted(
        required_dir_keys
        - set(dirs)
    )

    if missing_dir_keys:
        raise KeyError(
            "Dataset verification is missing DIRS keys: "
            f"{missing_dir_keys}"
        )

    validation = config[
        "dataset_validation"
    ]

    required_validation_keys = {
        "minimum_video_files",
        "minimum_labelled_images",
        "image_extensions",
        "video_extensions",
    }

    missing_validation_keys = sorted(
        required_validation_keys
        - set(validation)
    )

    if missing_validation_keys:
        raise KeyError(
            "Dataset verification is missing CONFIG keys: "
            f"{missing_validation_keys}"
        )

    raw_data_dir = Path(
        dirs[
            "raw_data_dir"
        ]
    )

    dataset_root_dir = Path(
        dirs[
            "dataset_root_dir"
        ]
    )

    labelled_images_dir = Path(
        dirs[
            "labelled_images_dir"
        ]
    )

    metadata_path = Path(
        dirs[
            "metadata_path"
        ]
    )

    image_extensions = frozenset(
        str(extension).casefold()
        for extension in validation[
            "image_extensions"
        ]
    )

    video_extensions = frozenset(
        str(extension).casefold()
        for extension in validation[
            "video_extensions"
        ]
    )

    minimum_video_files = int(
        validation[
            "minimum_video_files"
        ]
    )

    minimum_labelled_images = int(
        validation[
            "minimum_labelled_images"
        ]
    )

    if minimum_video_files < 1:
        raise ValueError(
            "minimum_video_files must be positive."
        )

    if minimum_labelled_images < 1:
        raise ValueError(
            "minimum_labelled_images must be positive."
        )

    raw_data_dir_exists = (
        raw_data_dir.is_dir()
    )

    dataset_root_dir_exists = (
        dataset_root_dir.is_dir()
    )

    labelled_images_dir_exists = (
        labelled_images_dir.is_dir()
    )

    metadata_available = (
        metadata_path.is_file()
    )

    video_count = (
        sum(
            1
            for path
            in dataset_root_dir.rglob("*")
            if (
                path.is_file()
                and path.suffix.casefold()
                in video_extensions
            )
        )
        if dataset_root_dir_exists
        else 0
    )

    labelled_image_count = (
        sum(
            1
            for path
            in labelled_images_dir.rglob("*")
            if (
                path.is_file()
                and path.suffix.casefold()
                in image_extensions
            )
        )
        if labelled_images_dir_exists
        else 0
    )

    minimum_video_count_met = (
        video_count
        >= minimum_video_files
    )

    minimum_labelled_image_count_met = (
        labelled_image_count
        >= minimum_labelled_images
    )

    validation_passed = all(
        [
            raw_data_dir_exists,
            dataset_root_dir_exists,
            labelled_images_dir_exists,
            metadata_available,
            minimum_video_count_met,
            minimum_labelled_image_count_met,
        ]
    )

    return {
        "raw_data_dir":
            str(raw_data_dir),

        "dataset_root_dir":
            str(dataset_root_dir),

        "labelled_images_dir":
            str(labelled_images_dir),

        "metadata_path":
            str(metadata_path),

        "raw_data_dir_exists":
            raw_data_dir_exists,

        "dataset_root_dir_exists":
            dataset_root_dir_exists,

        "labelled_images_dir_exists":
            labelled_images_dir_exists,

        "metadata_available":
            metadata_available,

        "video_count":
            video_count,

        "minimum_video_files":
            minimum_video_files,

        "minimum_video_count_met":
            minimum_video_count_met,

        "labelled_image_count":
            labelled_image_count,

        "minimum_labelled_images":
            minimum_labelled_images,

        "minimum_labelled_image_count_met":
            minimum_labelled_image_count_met,

        "validation_passed":
            validation_passed,
    }


def verify_raw_dataset(
    config,
    dirs,
):
    """
    Returns True when the raw Kvasir-Capsule dataset
    satisfies all declared minimum requirements.
    """

    inventory = (
        build_raw_dataset_inventory(
            config=config,
            dirs=dirs,
        )
    )

    return bool(
        inventory[
            "validation_passed"
        ]
    )




In [ ]:
def download_dataset_if_needed(
    config,
    dirs,
):
    """
    Downloads Kvasir-Capsule when the minimum raw-dataset
    validation checks do not pass.

    Side effect:
        May download files into dirs["raw_data_dir"].

    Returns:
        "already_available" or "downloaded".
    """

    if verify_raw_dataset(config, dirs):
        print(
            "Raw-dataset validation passed. "
            "Skipping download."
        )
        return "already_available"

    if not config["dataset_download_enabled"]:
        raise RuntimeError(
            "Kvasir-Capsule is missing or incomplete, "
            "and automatic download is disabled."
        )

    if shutil.which("osf") is None:
        raise RuntimeError(
            "The 'osf' command is not installed or is not "
            "available on PATH. Install osfclient first."
        )

    raw_data_dir = dirs["raw_data_dir"]

    raw_data_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    command = [
        "osf",
        "-p",
        config["dataset_osf_project_id"],
        "clone",
        str(raw_data_dir),
    ]

    print(
        "Downloading Kvasir-Capsule to persistent storage..."
    )

    try:
        subprocess.run(
            command,
            check=True,
        )

    except subprocess.CalledProcessError as error:
        raise RuntimeError(
            "The OSF download command failed."
        ) from error

    if not verify_raw_dataset(config, dirs):
        raise RuntimeError(
            "The download command completed, but the "
            "downloaded dataset did not pass validation."
        )

    print(
        "Dataset download completed and validation passed."
    )

    return "downloaded"


# ------------------------------------------------------------------
# Ensure that the raw dataset is available
# ------------------------------------------------------------------

DOWNLOAD_STATUS = (
    download_dataset_if_needed(
        config=CONFIG,
        dirs=DIRS,
    )
)


# ------------------------------------------------------------------
# Build the final inventory after the optional download
# ------------------------------------------------------------------

RAW_DATASET_INVENTORY = (
    build_raw_dataset_inventory(
        config=CONFIG,
        dirs=DIRS,
    )
)


RAW_DATASET_AVAILABLE = bool(
    RAW_DATASET_INVENTORY[
        "validation_passed"
    ]
)


if not RAW_DATASET_AVAILABLE:
    raise RuntimeError(
        "The final raw-dataset inventory did not pass "
        "validation after dataset preparation."
    )


raw_dataset_inventory_report = (
    pd.DataFrame.from_records(
        [
            RAW_DATASET_INVENTORY
        ]
    )
)


print(
    "Download status:",
    DOWNLOAD_STATUS,
)


print(
    "Raw dataset available:",
    RAW_DATASET_AVAILABLE,
)


display(
    raw_dataset_inventory_report.T.rename(
        columns={
            0:
                "value"
        }
    )
)

In [ ]:
# ------------------------------------------------------------------
# Raw-dataset file discovery
# ------------------------------------------------------------------

VALIDATION = CONFIG[
    "dataset_validation"
]


IMAGE_EXTENSIONS = frozenset(
    str(extension).casefold()
    for extension in VALIDATION[
        "image_extensions"
    ]
)


VIDEO_EXTENSIONS = frozenset(
    str(extension).casefold()
    for extension in VALIDATION[
        "video_extensions"
    ]
)


def discover_files(
    root,
    extensions,
):
    """
    Returns a deterministic tuple of physical files whose
    extensions match the declared allowed extensions.

    The filesystem is inspected but not modified.
    """

    root = Path(
        root
    )

    if not root.is_dir():
        raise NotADirectoryError(
            "Dataset directory does not exist: "
            f"{root}"
        )

    allowed_extensions = frozenset(
        str(extension).casefold()
        for extension in extensions
    )

    if not allowed_extensions:
        raise ValueError(
            "At least one file extension must be declared."
        )

    return tuple(
        sorted(
            (
                path
                for path in root.rglob("*")
                if (
                    path.is_file()
                    and path.suffix.casefold()
                    in allowed_extensions
                )
            ),
            key=lambda path:
                path.as_posix().casefold(),
        )
    )


def discover_raw_dataset_files(
    config,
    dirs,
):
    """
    Resolves the declared raw dataset paths and discovers
    labelled images and source videos.

    Returns a structured registry without modifying
    the filesystem.
    """

    required_dir_keys = {
        "dataset_root_dir",
        "labelled_images_dir",
        "metadata_path",
    }

    missing_dir_keys = sorted(
        required_dir_keys
        - set(dirs)
    )

    if missing_dir_keys:
        raise KeyError(
            "Raw-file discovery is missing DIRS keys: "
            f"{missing_dir_keys}"
        )

    validation = config[
        "dataset_validation"
    ]

    dataset_root_dir = Path(
        dirs[
            "dataset_root_dir"
        ]
    )

    labelled_images_dir = Path(
        dirs[
            "labelled_images_dir"
        ]
    )

    metadata_path = Path(
        dirs[
            "metadata_path"
        ]
    )

    if not metadata_path.is_file():
        raise FileNotFoundError(
            "The declared metadata file does not exist: "
            f"{metadata_path}"
        )

    labelled_image_files = discover_files(
        root=labelled_images_dir,
        extensions=validation[
            "image_extensions"
        ],
    )

    source_video_files = discover_files(
        root=dataset_root_dir,
        extensions=validation[
            "video_extensions"
        ],
    )

    return {
        "metadata_path":
            metadata_path,

        "image_files":
            labelled_image_files,

        "video_files":
            source_video_files,
    }


RAW_DATASET_FILES = (
    discover_raw_dataset_files(
        config=CONFIG,
        dirs=DIRS,
    )
)


METADATA_PATH = (
    RAW_DATASET_FILES[
        "metadata_path"
    ]
)


image_files = (
    RAW_DATASET_FILES[
        "image_files"
    ]
)


video_files = (
    RAW_DATASET_FILES[
        "video_files"
    ]
)


print(
    "Discovered labelled images:",
    f"{len(image_files):,}",
)


print(
    "Discovered videos:",
    f"{len(video_files):,}",
)


print(
    "Metadata path:",
    METADATA_PATH,
)

In [ ]:
if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        "The expected metadata file does not exist: "
        f"{METADATA_PATH}"
    )


df_raw = pd.read_csv(
    METADATA_PATH
)


print(
    "Discovered labelled images:",
    len(image_files),
)

print(
    "Discovered videos:",
    len(video_files),
)

print(
    "Metadata path:",
    METADATA_PATH,
)

print(
    "Metadata shape:",
    df_raw.shape,
)

display(
    df_raw.head()
)

print(
    "Raw metadata columns:",
    df_raw.columns.tolist(),
)

### 6. Normalization layer

In [ ]:
COLUMN_NAME_RULES = (
    (re.compile(r"[^a-z0-9]+"), "_"),
    (re.compile(r"_+"), "_"),
)

LABEL_RULES = (
    (re.compile(r"[_\-/]+"), " "),
    (re.compile(r"\s+"), " "),
)

COLUMN_ALIASES = {
    "file_name": "filename",
    "image_name": "filename",
    "image_filename": "filename",
    "video": "video_id",
    "video_name": "video_id",
    "frame": "frame_number",
    "frame_no": "frame_number",
    "label": "finding_class",
    "class": "finding_class",
    "category": "finding_category",
}

BBOX_SPEC = {
    "x_columns": ("x1", "x2", "x3", "x4"),
    "y_columns": ("y1", "y2", "y3", "y4"),
}

REQUIRED_COLUMNS = (
    "filename",
    "video_id",
    "finding_class",
)


def is_missing_scalar(value):
    """Returns True only for scalar missing values."""

    if value is None:
        return True

    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False

    return isinstance(
        missing,
        (bool, np.bool_),
    ) and bool(missing)


def scalar_to_text(value):
    """
    Converts a scalar metadata value to clean text.

    Sequences are rejected because Phase 2 metadata fields are
    expected to be scalar. This prevents silent label loss.
    """

    if isinstance(value, (list, tuple)):
        raise TypeError(
            "Expected a scalar metadata value, "
            f"but received {type(value).__name__}."
        )

    if is_missing_scalar(value):
        return ""

    return str(value).strip()


def apply_text_rules(
    text,
    rules,
):
    """Applies ordered regex transformations."""

    result = text

    for pattern, replacement in rules:
        result = pattern.sub(
            replacement,
            result,
        )

    return result


def apply_series_rules(
    series,
    rules,
):
    """Vectorized equivalent for a pandas Series."""

    result = series

    for pattern, replacement in rules:
        result = result.str.replace(
            pattern,
            replacement,
            regex=True,
        )

    return result


def normalize_column_name(value):
    """Creates a canonical snake_case column name."""

    text = scalar_to_text(value).casefold()

    return apply_text_rules(
        text=text,
        rules=COLUMN_NAME_RULES,
    ).strip("_")

def normalize_id(value):
    """Creates a filename-independent matching key."""

    text = Path(
        scalar_to_text(value)
    ).stem

    return re.sub(
        pattern=r"[^a-zA-Z0-9]+",
        repl="",
        string=text,
    ).casefold()

In [ ]:
def canonicalize_columns(
    dataframe,
    aliases,
):
    """
    Returns a new DataFrame with normalized and canonical columns.

    Raises an error if multiple source columns resolve to the
    same canonical column.
    """

    normalized_columns = [
        normalize_column_name(column)
        for column in dataframe.columns
    ]

    canonical_columns = [
        aliases.get(column, column)
        for column in normalized_columns
    ]

    canonical_index = pd.Index(
        canonical_columns
    )

    duplicate_columns = (
        canonical_index[
            canonical_index.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_columns:
        raise ValueError(
            "Multiple metadata columns resolve to the same "
            f"canonical name: {duplicate_columns}"
        )

    result = dataframe.copy()
    result.columns = canonical_columns

    return result


def validate_required_columns(
    dataframe,
    required_columns,
):
    """Validates the input schema without modifying it."""

    missing_columns = sorted(
        set(required_columns)
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Missing required metadata columns: "
            f"{missing_columns}"
        )

    return dataframe

In [ ]:
def add_label_columns(
    dataframe,
    clinical_group_map,
    strict=True,
):
    """
    Adds clean, normalized, and grouped label columns.

    The source label is preserved in finding_class_raw.
    """

    source_labels = (
        dataframe["finding_class_raw"]
        if "finding_class_raw" in dataframe.columns
        else dataframe["finding_class"]
    )

    clean_labels = source_labels.map(
        scalar_to_text
    )

    normalized_labels = apply_series_rules(
        series=(
            clean_labels
            .str.casefold()
            .str.strip()
        ),
        rules=LABEL_RULES,
    ).str.strip()

    clinical_groups = normalized_labels.map(
        clinical_group_map
    )

    unmapped_labels = sorted(
        normalized_labels[
            normalized_labels.ne("")
            & clinical_groups.isna()
        ]
        .unique()
        .tolist()
    )

    if strict and unmapped_labels:
        raise ValueError(
            "Labels missing from clinical_group_map: "
            f"{unmapped_labels}"
        )

    return dataframe.assign(
        finding_class_raw=source_labels,
        finding_class=clean_labels,
        finding_class_normalized=normalized_labels,
        clinical_group=clinical_groups,
    )


def add_matching_keys(dataframe):
    """Adds normalized image and video matching keys."""

    return dataframe.assign(
        image_key=(
            dataframe["filename"]
            .map(normalize_id)
        ),
        video_key=(
            dataframe["video_id"]
            .map(normalize_id)
        ),
    )

In [ ]:
def add_empty_bbox_columns(dataframe):
    """Returns the stable bbox schema for data without annotations."""

    index = dataframe.index

    return dataframe.assign(
        bbox_annotation_present=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_complete=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_invalid=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        has_bbox=pd.Series(
            False,
            index=index,
            dtype=bool,
        ),
        bbox_xmin=np.nan,
        bbox_ymin=np.nan,
        bbox_xmax=np.nan,
        bbox_ymax=np.nan,
        bbox_width=np.nan,
        bbox_height=np.nan,
        bbox_area=np.nan,
    )


def normalize_bounding_boxes(
    dataframe,
    bbox_spec,
):
    """
    Adds a validated axis-aligned bounding-box representation.

    Original coordinate columns are not overwritten.
    """

    x_columns = tuple(
        bbox_spec["x_columns"]
    )

    y_columns = tuple(
        bbox_spec["y_columns"]
    )

    expected_columns = (
        x_columns
        + y_columns
    )

    present_columns = tuple(
        column
        for column in expected_columns
        if column in dataframe.columns
    )

    if not present_columns:
        return add_empty_bbox_columns(
            dataframe
        )

    missing_schema_columns = sorted(
        set(expected_columns)
        - set(present_columns)
    )

    if missing_schema_columns:
        raise ValueError(
            "Incomplete bounding-box schema. "
            f"Missing columns: {missing_schema_columns}"
        )

    coordinates = (
        dataframe
        .loc[:, expected_columns]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    annotation_present = (
        coordinates
        .notna()
        .any(axis=1)
    )

    bbox_complete = (
        coordinates
        .notna()
        .all(axis=1)
    )

    xmin = coordinates[
        list(x_columns)
    ].min(axis=1)

    xmax = coordinates[
        list(x_columns)
    ].max(axis=1)

    ymin = coordinates[
        list(y_columns)
    ].min(axis=1)

    ymax = coordinates[
        list(y_columns)
    ].max(axis=1)

    geometry_valid = (
        bbox_complete
        & xmax.gt(xmin)
        & ymax.gt(ymin)
    )

    width = (
        xmax - xmin
    ).where(geometry_valid)

    height = (
        ymax - ymin
    ).where(geometry_valid)

    return dataframe.assign(
        bbox_annotation_present=annotation_present,
        bbox_complete=bbox_complete,
        bbox_invalid=(
            annotation_present
            & ~geometry_valid
        ),
        has_bbox=geometry_valid,
        bbox_xmin=xmin.where(geometry_valid),
        bbox_ymin=ymin.where(geometry_valid),
        bbox_xmax=xmax.where(geometry_valid),
        bbox_ymax=ymax.where(geometry_valid),
        bbox_width=width,
        bbox_height=height,
        bbox_area=width * height,
    )

In [ ]:
def normalize_metadata(
    dataframe,
    clinical_group_map,
):
    """
    Pure-by-contract Phase 2 metadata normalization pipeline.

    Input:
        raw DataFrame

    Output:
        new normalized DataFrame
    """

    return (
        dataframe
        .pipe(
            canonicalize_columns,
            aliases=COLUMN_ALIASES,
        )
        .pipe(
            validate_required_columns,
            required_columns=REQUIRED_COLUMNS,
        )
        .pipe(
            add_label_columns,
            clinical_group_map=clinical_group_map,
            strict=True,
        )
        .pipe(
            add_matching_keys,
        )
        .pipe(
            normalize_bounding_boxes,
            bbox_spec=BBOX_SPEC,
        )
    )


df = normalize_metadata(
    dataframe=df_raw,
    clinical_group_map=(
        CONFIG["clinical_group_map"]
    ),
)

### 7. Dataset validation

In [ ]:
DATASET_CHARACTERISTIC_REPORT_COLUMNS = [
    "characteristic",
    "actual_value",
    "expected_value",
    "validation_passed",
]


def build_dataset_characteristics_report(
    dataframe,
    config,
):
    """
    Builds a declarative validation report for the
    expected Kvasir-Capsule dataset characteristics.

    The input DataFrame is not modified.
    """

    required_columns = {
        "image_key",
        "video_key",
        "frame_number",
        "finding_class_normalized",
    }

    missing_columns = sorted(
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Dataset validation is missing columns: "
            f"{missing_columns}"
        )

    identity_columns = [
        "image_key",
        "video_key",
        "finding_class_normalized",
    ]

    invalid_identity_mask = (
        dataframe[
            identity_columns
        ]
        .astype("string")
        .apply(
            lambda column:
                column.isna()
                | column.str.strip().eq("")
        )
        .any(axis=1)
    )

    if invalid_identity_mask.any():
        raise ValueError(
            "Normalized metadata contains "
            f"{int(invalid_identity_mask.sum()):,} "
            "rows with missing identity values."
        )

    frame_numbers = pd.to_numeric(
        dataframe[
            "frame_number"
        ],
        errors="coerce",
    )

    invalid_frame_number_mask = (
        frame_numbers.isna()
        | frame_numbers.lt(0)
        | frame_numbers.mod(1).ne(0)
    )

    if invalid_frame_number_mask.any():
        raise ValueError(
            "Normalized metadata contains "
            f"{int(invalid_frame_number_mask.sum()):,} "
            "invalid frame numbers."
        )

    actual_values = pd.Series(
        {
            "metadata_rows":
                len(dataframe),

            "unique_labelled_images":
                dataframe[
                    "image_key"
                ].nunique(
                    dropna=True
                ),

            "unique_labelled_video_frames":
                (
                    dataframe
                    .assign(
                        frame_number=(
                            frame_numbers.astype("Int64")
                        )
                    )
                    [
                        [
                            "video_key",
                            "frame_number",
                        ]
                    ]
                    .drop_duplicates()
                    .shape[0]
                ),

            "finding_classes":
                dataframe[
                    "finding_class_normalized"
                ].nunique(
                    dropna=True
                ),

            "labelled_videos":
                dataframe[
                    "video_key"
                ].nunique(
                    dropna=True
                ),
        },
        dtype="int64",
        name="actual_value",
    )

    expected_values = pd.Series(
        {
            "metadata_rows":
                int(
                    config[
                        "expected_labelled_frames"
                    ]
                ),

            "unique_labelled_images":
                int(
                    config[
                        "expected_labelled_frames"
                    ]
                ),

            "unique_labelled_video_frames":
                int(
                    config[
                        "expected_labelled_frames"
                    ]
                ),

            "finding_classes":
                int(
                    config[
                        "expected_classes"
                    ]
                ),

            "labelled_videos":
                int(
                    config[
                        "expected_labelled_videos"
                    ]
                ),
        },
        dtype="int64",
        name="expected_value",
    )

    return (
        pd.concat(
            [
                actual_values,
                expected_values,
            ],
            axis=1,
        )
        .rename_axis(
            "characteristic"
        )
        .reset_index()
        .assign(
            validation_passed=lambda data:
                data[
                    "actual_value"
                ].eq(
                    data[
                        "expected_value"
                    ]
                )
        )
        .loc[
            :,
            DATASET_CHARACTERISTIC_REPORT_COLUMNS,
        ]
    )


def validate_dataset_characteristics(
    validation_report,
):
    """
    Raises ValueError when any expected dataset
    characteristic does not match.
    """

    required_columns = set(
        DATASET_CHARACTERISTIC_REPORT_COLUMNS
    )

    missing_columns = sorted(
        required_columns
        - set(validation_report.columns)
    )

    if missing_columns:
        raise KeyError(
            "Dataset-characteristics report is missing "
            f"columns: {missing_columns}"
        )

    failed_checks = (
        validation_report.loc[
            ~validation_report[
                "validation_passed"
            ],
            [
                "characteristic",
                "actual_value",
                "expected_value",
            ],
        ]
    )

    if not failed_checks.empty:
        raise ValueError(
            "Dataset characteristic validation failed: "
            f"{failed_checks.to_dict(orient='records')}"
        )

    return validation_report


dataset_characteristics_report = (
    build_dataset_characteristics_report(
        dataframe=df,
        config=CONFIG,
    )
)


display(
    dataset_characteristics_report
)


dataset_characteristics_report = (
    dataset_characteristics_report
    .pipe(
        validate_dataset_characteristics
    )
)


print(
    "PASS: all expected dataset characteristics "
    "were validated."
)

### 8. Map labelled images and videos to files



In [ ]:
def build_file_index(
    files,
):
    """
    Builds a lookup index from normalized file IDs
    to physical file paths.
    """

    index = defaultdict(list)

    for path in files:
        index[
            normalize_id(path.name)
        ].append(path)

    return index


def resolve_unique_file(
    key,
    file_index,
):
    """
    Returns the file path when exactly one file
    matches the normalized identifier.
    """

    matches = file_index.get(
        key,
        [],
    )

    return (
        matches[0]

        if len(matches) == 1
        else None
    )

image_index = build_file_index(
    image_files
)


df["image_path"] = (
    df["image_key"]
    .map(
        lambda key: resolve_unique_file(
            key,
            image_index,
        )
    )
)


df["image_exists"] = (
    df["image_path"]
    .notna()
)

video_index = build_file_index(
    video_files
)


df["video_path"] = (
    df["video_key"]
    .map(
        lambda key: resolve_unique_file(
            key,
            video_index,
        )
    )
)


df["video_exists"] = (
    df["video_path"]
    .notna()
)

print(
    "Metadata rows with resolved video:",
    df["video_exists"].sum(),
    "/",
    len(df),
)

print(
    "Unique metadata rows with resolved video:",
    df.loc[
        df["video_exists"],
        "video_key",
    ].nunique(),
)


def find_index_collisions(file_index):
    """
    Returns normalized IDs associated with multiple files.
    """

    return {
        key: tuple(paths)
        for key, paths in file_index.items()
        if len(paths) > 1
    }


image_collisions = find_index_collisions(
    image_index
)

video_collisions = find_index_collisions(
    video_index
)


if image_collisions:
    raise ValueError(
        "Ambiguous normalized image IDs found: "
        f"{list(image_collisions)[:10]}"
    )

if video_collisions:
    raise ValueError(
        "Ambiguous normalized video IDs found: "
        f"{list(video_collisions)[:10]}"
    )

### 9. Probe video metadata

### 10. Build video manifest

In [ ]:
# ------------------------------------------------------------------
# Probe video metadata
# ------------------------------------------------------------------

VIDEO_PROBE_COLUMNS = [
    "container_opened",
    "first_frame_readable",
    "container_fps",
    "frame_count",
    "width",
    "height",
    "container_duration_seconds",
]


def empty_video_probe():
    """
    Returns the stable technical schema for an
    unreadable video container.
    """

    return {
        "container_opened":
            False,

        "first_frame_readable":
            False,

        "container_fps":
            np.nan,

        "frame_count":
            np.nan,

        "width":
            np.nan,

        "height":
            np.nan,

        "container_duration_seconds":
            np.nan,
    }


def probe_video(
    video_path,
):
    """
    Reads technical metadata from one video container.

    The reported FPS and duration describe the exported
    video container, not necessarily the original clinical
    capture timeline.

    The source file is not modified.
    """

    video_path = Path(
        video_path
    )

    cap = cv2.VideoCapture(
        str(video_path)
    )

    try:
        if not cap.isOpened():
            return empty_video_probe()

        raw_fps = cap.get(
            cv2.CAP_PROP_FPS
        )

        raw_frame_count = cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )

        raw_width = cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )

        raw_height = cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )

        first_frame_readable, _ = (
            cap.read()
        )

    finally:
        cap.release()

    container_fps = (
        float(raw_fps)
        if (
            np.isfinite(raw_fps)
            and raw_fps > 0
        )
        else np.nan
    )

    frame_count = (
        int(raw_frame_count)
        if (
            np.isfinite(raw_frame_count)
            and raw_frame_count > 0
            and float(
                raw_frame_count
            ).is_integer()
        )
        else np.nan
    )

    width = (
        int(raw_width)
        if (
            np.isfinite(raw_width)
            and raw_width > 0
            and float(
                raw_width
            ).is_integer()
        )
        else np.nan
    )

    height = (
        int(raw_height)
        if (
            np.isfinite(raw_height)
            and raw_height > 0
            and float(
                raw_height
            ).is_integer()
        )
        else np.nan
    )

    container_duration_seconds = (
        float(
            frame_count
            / container_fps
        )
        if (
            np.isfinite(frame_count)
            and np.isfinite(
                container_fps
            )
        )
        else np.nan
    )

    return {
        "container_opened":
            True,

        "first_frame_readable":
            bool(
                first_frame_readable
            ),

        "container_fps":
            container_fps,

        "frame_count":
            frame_count,

        "width":
            width,

        "height":
            height,

        "container_duration_seconds":
            container_duration_seconds,
    }

In [ ]:
# ------------------------------------------------------------------
# Build video manifest
# ------------------------------------------------------------------

VIDEO_MANIFEST_COLUMNS = [
    "video_key",
    "video_filename",
    "video_path",
    "video_relative_path",
    "video_annotation_type",
    "container_opened",
    "first_frame_readable",
    "container_fps",
    "frame_count",
    "width",
    "height",
    "container_duration_seconds",
]


def build_video_record(
    video_path,
    labelled_video_keys,
    dataset_root,
):
    """
    Builds one structured manifest record for one
    physical video file.

    Technical video metadata is read without modifying
    the source file.
    """

    video_path = Path(
        video_path
    )

    dataset_root = Path(
        dataset_root
    )

    video_key = normalize_id(
        video_path.name
    )

    if not video_key:
        raise ValueError(
            "Cannot build video record from an empty "
            f"normalized video key: {video_path}"
        )

    try:
        video_relative_path = (
            video_path
            .relative_to(
                dataset_root
            )
            .as_posix()
        )

    except ValueError as error:
        raise ValueError(
            "Video path is outside the declared dataset root. "
            f"Video: {video_path}. "
            f"Dataset root: {dataset_root}."
        ) from error

    video_probe = probe_video(
        video_path
    )

    is_labelled_video = (
        video_key
        in labelled_video_keys
    )

    return {
        "video_key":
            video_key,

        "video_filename":
            video_path.name,

        "video_path":
            str(video_path),

        "video_relative_path":
            video_relative_path,

        "video_annotation_type": (
            "partially_labelled"
            if is_labelled_video
            else "fully_unlabelled"
        ),

        **video_probe,
    }


def build_video_manifest(
    video_files,
    labelled_video_keys,
    dataset_root,
):
    """
    Builds a deterministic video-level manifest.

    One row represents one physical source video.
    """

    video_files = tuple(
        Path(video_path)
        for video_path in video_files
    )

    if not video_files:
        raise ValueError(
            "Cannot build a video manifest because no "
            "physical video files were provided."
        )

    duplicate_path_count = (
        len(video_files)
        - len(set(video_files))
    )

    if duplicate_path_count:
        raise ValueError(
            "The physical video inventory contains "
            f"{duplicate_path_count} duplicate paths."
        )

    records = [
        build_video_record(
            video_path=video_path,
            labelled_video_keys=(
                labelled_video_keys
            ),
            dataset_root=dataset_root,
        )

        for video_path in tqdm(
            video_files,
            desc="Building video manifest",
        )
    ]

    return (
        pd.DataFrame.from_records(
            records,
            columns=(
                VIDEO_MANIFEST_COLUMNS
            ),
        )
        .sort_values(
            "video_key",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


labelled_video_keys = frozenset(
    df[
        "video_key"
    ]
    .dropna()
    .astype("string")
    .str.strip()
    .loc[
        lambda values:
            values.ne("")
    ]
    .unique()
)


video_manifest = (
    build_video_manifest(
        video_files=video_files,
        labelled_video_keys=(
            labelled_video_keys
        ),
        dataset_root=(
            DIRS[
                "dataset_root_dir"
            ]
        ),
    )
)


print(
    "Video manifest rows:",
    f"{len(video_manifest):,}",
)


display(
    video_manifest.head()
)

In [ ]:
def validate_video_manifest(
    dataframe,
    labelled_video_keys,
    config,
):
    """
    Validates the complete physical video inventory,
    technical probe results, and consistency with
    labelled metadata.

    The input DataFrame is not modified.
    """

    required_columns = set(
        VIDEO_MANIFEST_COLUMNS
    )

    missing_columns = sorted(
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Video manifest is missing required columns: "
            f"{missing_columns}"
        )

    if dataframe.empty:
        raise ValueError(
            "Video manifest cannot be empty."
        )

    # --------------------------------------------------------------
    # Validate normalized video identifiers
    # --------------------------------------------------------------

    video_keys = (
        dataframe[
            "video_key"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_video_key_mask = (
        video_keys.isna()
        | video_keys.eq("")
    )

    if invalid_video_key_mask.any():
        raise ValueError(
            "Video manifest contains "
            f"{int(invalid_video_key_mask.sum())} "
            "missing or empty video keys."
        )

    duplicate_video_keys = (
        video_keys.loc[
            video_keys.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_video_keys:
        raise ValueError(
            "Multiple physical videos resolve to the same "
            f"video key: {duplicate_video_keys}"
        )

    # --------------------------------------------------------------
    # Validate physical and relative paths
    # --------------------------------------------------------------

    video_paths = (
        dataframe[
            "video_path"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_video_path_mask = (
        video_paths.isna()
        | video_paths.eq("")
    )

    if invalid_video_path_mask.any():
        raise ValueError(
            "Video manifest contains "
            f"{int(invalid_video_path_mask.sum())} "
            "missing video paths."
        )

    physical_file_exists = (
        video_paths.map(
            lambda value:
                Path(value).is_file()
        )
    )

    if not physical_file_exists.all():
        missing_files = (
            video_paths.loc[
                ~physical_file_exists
            ]
            .tolist()
        )

        raise FileNotFoundError(
            "Video manifest contains paths that do not "
            f"exist: {missing_files}"
        )

    duplicate_video_paths = (
        video_paths.loc[
            video_paths.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_video_paths:
        raise ValueError(
            "Duplicate physical video paths found: "
            f"{duplicate_video_paths}"
        )

    relative_paths = (
        dataframe[
            "video_relative_path"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_relative_path_mask = (
        relative_paths.isna()
        | relative_paths.eq("")
        | relative_paths.map(
            lambda value:
                (
                    Path(value).is_absolute()
                    if pd.notna(value)
                    else True
                )
        )
    )

    if invalid_relative_path_mask.any():
        raise ValueError(
            "Video manifest contains invalid or absolute "
            "video-relative paths."
        )

    if relative_paths.duplicated().any():
        raise ValueError(
            "Video manifest contains duplicate "
            "video-relative paths."
        )

    # --------------------------------------------------------------
    # Validate complete physical inventory
    # --------------------------------------------------------------

    actual_total_videos = len(
        dataframe
    )

    expected_total_videos = int(
        config[
            "expected_total_videos"
        ]
    )

    if actual_total_videos != expected_total_videos:
        raise ValueError(
            "Unexpected number of physical videos: "
            f"expected {expected_total_videos}, "
            f"found {actual_total_videos}."
        )

    # --------------------------------------------------------------
    # Validate annotation categories and counts
    # --------------------------------------------------------------

    annotation_types = (
        dataframe[
            "video_annotation_type"
        ]
        .astype("string")
        .str.strip()
    )

    expected_annotation_types = {
        "partially_labelled",
        "fully_unlabelled",
    }

    invalid_annotation_mask = (
        annotation_types.isna()
        | annotation_types.eq("")
        | ~annotation_types.isin(
            expected_annotation_types
        )
    )

    if invalid_annotation_mask.any():
        invalid_annotation_types = (
            annotation_types.loc[
                invalid_annotation_mask
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            "Unexpected or missing video annotation "
            f"types: {invalid_annotation_types}"
        )

    annotation_counts = (
        annotation_types.value_counts()
    )

    actual_labelled_videos = int(
        annotation_counts.get(
            "partially_labelled",
            0,
        )
    )

    actual_unlabelled_videos = int(
        annotation_counts.get(
            "fully_unlabelled",
            0,
        )
    )

    expected_labelled_videos = int(
        config[
            "expected_labelled_videos"
        ]
    )

    expected_unlabelled_videos = int(
        config[
            "expected_unlabelled_videos"
        ]
    )

    if (
        actual_labelled_videos
        != expected_labelled_videos
    ):
        raise ValueError(
            "Unexpected number of partially labelled "
            "videos: "
            f"expected {expected_labelled_videos}, "
            f"found {actual_labelled_videos}."
        )

    if (
        actual_unlabelled_videos
        != expected_unlabelled_videos
    ):
        raise ValueError(
            "Unexpected number of fully unlabelled "
            "videos: "
            f"expected {expected_unlabelled_videos}, "
            f"found {actual_unlabelled_videos}."
        )

    # --------------------------------------------------------------
    # Validate OpenCV probe results
    # --------------------------------------------------------------

    container_opened = (
        dataframe[
            "container_opened"
        ]
        .astype("boolean")
    )

    first_frame_readable = (
        dataframe[
            "first_frame_readable"
        ]
        .astype("boolean")
    )

    unreadable_container_mask = (
        container_opened.isna()
        | ~container_opened.fillna(False)
    )

    unreadable_first_frame_mask = (
        first_frame_readable.isna()
        | ~first_frame_readable.fillna(False)
    )

    if unreadable_container_mask.any():
        unreadable_videos = (
            dataframe.loc[
                unreadable_container_mask,
                "video_key",
            ]
            .tolist()
        )

        raise RuntimeError(
            "OpenCV could not open these videos: "
            f"{unreadable_videos}"
        )

    if unreadable_first_frame_mask.any():
        unreadable_videos = (
            dataframe.loc[
                unreadable_first_frame_mask,
                "video_key",
            ]
            .tolist()
        )

        raise RuntimeError(
            "OpenCV could not read the first frame of "
            f"these videos: {unreadable_videos}"
        )

    technical_metadata = (
        dataframe[
            [
                "container_fps",
                "frame_count",
                "width",
                "height",
                "container_duration_seconds",
            ]
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    invalid_fps_mask = (
        technical_metadata[
            "container_fps"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "container_fps"
            ]
        )
        | technical_metadata[
            "container_fps"
        ].le(0)
    )

    invalid_frame_count_mask = (
        technical_metadata[
            "frame_count"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "frame_count"
            ]
        )
        | technical_metadata[
            "frame_count"
        ].le(0)
        | technical_metadata[
            "frame_count"
        ].mod(1).ne(0)
    )

    invalid_width_mask = (
        technical_metadata[
            "width"
        ].isna()
        | technical_metadata[
            "width"
        ].le(0)
        | technical_metadata[
            "width"
        ].mod(1).ne(0)
    )

    invalid_height_mask = (
        technical_metadata[
            "height"
        ].isna()
        | technical_metadata[
            "height"
        ].le(0)
        | technical_metadata[
            "height"
        ].mod(1).ne(0)
    )

    invalid_duration_mask = (
        technical_metadata[
            "container_duration_seconds"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "container_duration_seconds"
            ]
        )
        | technical_metadata[
            "container_duration_seconds"
        ].le(0)
    )

    invalid_technical_mask = (
        invalid_fps_mask
        | invalid_frame_count_mask
        | invalid_width_mask
        | invalid_height_mask
        | invalid_duration_mask
    )

    if invalid_technical_mask.any():
        invalid_videos = (
            dataframe.loc[
                invalid_technical_mask,
                "video_key",
            ]
            .tolist()
        )

        raise ValueError(
            "Invalid technical video metadata found for: "
            f"{invalid_videos}"
        )

    # --------------------------------------------------------------
    # Validate expected container FPS
    # --------------------------------------------------------------

    expected_container_fps = float(
        config[
            "expected_export_container_fps"
        ]
    )

    fps_matches_expected = np.isclose(
        technical_metadata[
            "container_fps"
        ].to_numpy(
            dtype=float
        ),
        expected_container_fps,
        rtol=0.0,
        atol=1e-3,
    )

    if not fps_matches_expected.all():
        fps_mismatches = (
            dataframe.loc[
                ~fps_matches_expected,
                [
                    "video_key",
                    "container_fps",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Unexpected exported-container FPS values. "
            f"Expected {expected_container_fps}: "
            f"{fps_mismatches}"
        )

    # --------------------------------------------------------------
    # Validate total extractable frame inventory
    # --------------------------------------------------------------

    actual_total_frames = int(
        technical_metadata[
            "frame_count"
        ].sum()
    )

    expected_total_frames = int(
        config[
            "expected_total_extractable_frames"
        ]
    )

    if actual_total_frames != expected_total_frames:
        raise ValueError(
            "Unexpected total extractable frame count: "
            f"expected {expected_total_frames:,}, "
            f"found {actual_total_frames:,}."
        )

    # --------------------------------------------------------------
    # Cross-check labelled metadata against physical videos
    # --------------------------------------------------------------

    manifest_labelled_video_keys = frozenset(
        dataframe.loc[
            annotation_types.eq(
                "partially_labelled"
            ),
            "video_key",
        ]
    )

    metadata_labelled_video_keys = frozenset(
        labelled_video_keys
    )

    missing_physical_videos = sorted(
        metadata_labelled_video_keys
        - manifest_labelled_video_keys
    )

    unexpected_labelled_videos = sorted(
        manifest_labelled_video_keys
        - metadata_labelled_video_keys
    )

    if (
        missing_physical_videos
        or unexpected_labelled_videos
    ):
        raise ValueError(
            "Video manifest is inconsistent with metadata. "
            "Missing physical labelled videos: "
            f"{missing_physical_videos}. "
            "Unexpected labelled videos: "
            f"{unexpected_labelled_videos}."
        )

    return dataframe


video_manifest = (
    video_manifest
    .pipe(
        validate_video_manifest,
        labelled_video_keys=(
            labelled_video_keys
        ),
        config=CONFIG,
    )
)



video_counts = (
    video_manifest[
        "video_annotation_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "video_count"
    )
    .reset_index()
    .rename(
        columns={
            "video_annotation_type":
                "annotation_type"
        }
    )
)


video_inventory_summary = (
    pd.DataFrame.from_records(
        [
            {
                "total_videos":
                    len(video_manifest),

                "partially_labelled_videos":
                    int(
                        video_manifest[
                            "video_annotation_type"
                        ]
                        .eq(
                            "partially_labelled"
                        )
                        .sum()
                    ),

                "fully_unlabelled_videos":
                    int(
                        video_manifest[
                            "video_annotation_type"
                        ]
                        .eq(
                            "fully_unlabelled"
                        )
                        .sum()
                    ),

                "total_extractable_frames":
                    int(
                        video_manifest[
                            "frame_count"
                        ]
                        .sum()
                    ),

                "container_fps_min":
                    float(
                        video_manifest[
                            "container_fps"
                        ]
                        .min()
                    ),

                "container_fps_max":
                    float(
                        video_manifest[
                            "container_fps"
                        ]
                        .max()
                    ),
            }
        ]
    )
)


display(
    video_inventory_summary
)


display(
    video_counts
)

### 11. Verify frame-number alignment

In [ ]:
# ------------------------------------------------------------------
# Video-frame alignment
# ------------------------------------------------------------------

def read_video_frame(
    video_path,
    frame_index,
):
    """
    Reads one OpenCV-indexed frame from a video.

    Returns:
        BGR image array when successful.
        None when the frame cannot be read.
    """

    try:
        frame_index = int(
            frame_index
        )

    except (TypeError, ValueError):
        return None

    if frame_index < 0:
        return None

    cap = cv2.VideoCapture(
        str(video_path)
    )

    try:
        if not cap.isOpened():
            return None

        seek_succeeded = cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            frame_index,
        )

        if not seek_succeeded:
            return None

        success, frame = cap.read()

        return (
            frame
            if success
            else None
        )

    finally:
        cap.release()


def image_similarity(
    image_a,
    image_b,
    comparison_size,
):
    """
    Computes grayscale structural similarity between two
    images after resizing them to a common resolution.
    """

    if image_a is None or image_b is None:
        return np.nan

    comparison_size = tuple(
        int(value)
        for value in comparison_size
    )

    if (
        len(comparison_size) != 2
        or any(
            value <= 0
            for value in comparison_size
        )
    ):
        raise ValueError(
            "comparison_size must contain two positive "
            f"integers, found: {comparison_size}"
        )

    gray_a = cv2.cvtColor(
        image_a,
        cv2.COLOR_BGR2GRAY,
    )

    gray_b = cv2.cvtColor(
        image_b,
        cv2.COLOR_BGR2GRAY,
    )

    resized_a = cv2.resize(
        gray_a,
        comparison_size,
    )

    resized_b = cv2.resize(
        gray_b,
        comparison_size,
    )

    return float(
        ssim(
            resized_a,
            resized_b,
            data_range=255,
        )
    )


def infer_frame_index_offset(
    dataframe,
    config,
):
    """
    Infers the alignment between Kvasir metadata frame numbers
    and zero-based OpenCV video-frame indices.

    One candidate frame is sampled from each selected video
    to avoid overrepresenting videos with many labelled frames.

    Returns:
        selected_offset
        alignment_scores
    """

    required_columns = {
        "image_exists",
        "video_exists",
        "image_path",
        "video_path",
        "video_key",
        "frame_number",
    }

    missing_columns = sorted(
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Frame-alignment input is missing columns: "
            f"{missing_columns}"
        )


    # --------------------------------------------------------------
    # Validate and normalize frame numbers
    # --------------------------------------------------------------

    frame_numbers = pd.to_numeric(
        dataframe["frame_number"],
        errors="coerce",
    )

    valid_frame_numbers = (
        frame_numbers.notna()
        & frame_numbers.ge(0)
        & frame_numbers.eq(
            np.floor(frame_numbers)
        )
    )

    eligible_rows = (
        dataframe["image_exists"]
        & dataframe["video_exists"]
        & valid_frame_numbers
    )

    candidates = (
        dataframe.loc[
            eligible_rows
        ]
        .assign(
            alignment_frame_number=(
                frame_numbers.loc[
                    eligible_rows
                ]
                .astype("int64")
            )
        )
    )

    if candidates.empty:
        raise RuntimeError(
            "No valid labelled image/video pairs are available "
            "for frame-alignment verification."
        )


    # --------------------------------------------------------------
    # Sample across distinct labelled videos
    # --------------------------------------------------------------

    shuffled_candidates = candidates.sample(
        frac=1,
        random_state=config["seed"],
    )

    unique_video_candidates = (
        shuffled_candidates
        .drop_duplicates(
            subset="video_key"
        )
    )

    sample_size = min(
        config["frame_alignment_sample_size"],
        len(unique_video_candidates),
    )

    sampled_examples = (
        unique_video_candidates
        .head(sample_size)
    )

    if sampled_examples.empty:
        raise RuntimeError(
            "No frame-alignment examples could be sampled."
        )


    # --------------------------------------------------------------
    # Evaluate configured candidate offsets
    # --------------------------------------------------------------

    candidate_offsets = tuple(
        int(offset)
        for offset in config[
            "frame_index_offset_candidates"
        ]
    )

    if not candidate_offsets:
        raise ValueError(
            "No frame-index offset candidates are configured."
        )

    comparison_size = config[
        "frame_alignment_comparison_size"
    ]

    alignment_scores = {}

    for offset in candidate_offsets:
        scores = []

        for _, example in (
            sampled_examples.iterrows()
        ):
            labelled_image = cv2.imread(
                str(example["image_path"])
            )

            video_frame = read_video_frame(
                video_path=example["video_path"],
                frame_index=(
                    example[
                        "alignment_frame_number"
                    ]
                    + offset
                ),
            )

            score = image_similarity(
                image_a=labelled_image,
                image_b=video_frame,
                comparison_size=comparison_size,
            )

            if np.isfinite(score):
                scores.append(
                    float(score)
                )

        alignment_scores[offset] = {
            "mean_ssim": (
                float(np.mean(scores))
                if scores
                else np.nan
            ),

            "median_ssim": (
                float(np.median(scores))
                if scores
                else np.nan
            ),

            "valid_pairs":
                len(scores),

            "sampled_pairs":
                len(sampled_examples),
        }


    # --------------------------------------------------------------
    # Select the best sufficiently supported offset
    # --------------------------------------------------------------

    minimum_valid_fraction = float(
        config[
            "frame_alignment_min_valid_fraction"
        ]
    )

    if not (
        0 < minimum_valid_fraction <= 1
    ):
        raise ValueError(
            "frame_alignment_min_valid_fraction must be "
            "within the interval (0, 1]."
        )

    minimum_valid_pairs = max(
        1,
        int(
            np.ceil(
                minimum_valid_fraction
                * len(sampled_examples)
            )
        ),
    )

    valid_scores = {
        offset: statistics["mean_ssim"]
        for offset, statistics
        in alignment_scores.items()
        if (
            statistics["valid_pairs"]
            >= minimum_valid_pairs
            and np.isfinite(
                statistics["mean_ssim"]
            )
        )
    }

    if not valid_scores:
        raise RuntimeError(
            "Frame alignment could not be evaluated with "
            "enough valid image/video pairs for any offset."
        )

    selected_offset = max(
        valid_scores,
        key=valid_scores.get,
    )

    return (
        selected_offset,
        alignment_scores,
    )


FRAME_INDEX_OFFSET, FRAME_ALIGNMENT_SCORES = (
    infer_frame_index_offset(
        dataframe=df,
        config=CONFIG,
    )
)


frame_alignment_report = (
    pd.DataFrame
    .from_dict(
        FRAME_ALIGNMENT_SCORES,
        orient="index",
    )
    .rename_axis(
        "offset"
    )
    .reset_index()
    .sort_values(
        "mean_ssim",
        ascending=False,
    )
)


display(
    frame_alignment_report
)

print(
    "Selected frame index offset:",
    FRAME_INDEX_OFFSET,
)

### 12. Audit clinical label taxonomy





In [ ]:
# ------------------------------------------------------------------
# Clinical label taxonomy audit
# ------------------------------------------------------------------

clinical_taxonomy_report = (
    df[
        [
            "finding_class",
            "finding_class_normalized",
            "clinical_group",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "clinical_group",
            "finding_class_normalized",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "PASS: all non-empty finding classes were mapped "
    "to a clinical group during metadata normalization."
)

print(
    "Finding classes:",
    clinical_taxonomy_report[
        "finding_class_normalized"
    ].nunique(),
)

print(
    "Clinical groups:",
    clinical_taxonomy_report[
        "clinical_group"
    ].nunique(),
)

display(
    clinical_taxonomy_report
)

### 13. Build labelled-frame manifest

In [ ]:
# ------------------------------------------------------------------
# Portable path handling
# ------------------------------------------------------------------

def path_relative_to_storage(
    value,
    storage_root,
):
    """
    Converts a physical path into a POSIX path relative to
    storage_root.

    Raises an error when the physical path is outside the
    configured storage root.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None

    except TypeError:
        pass

    storage_root = Path(
        storage_root
    ).resolve()

    path = Path(
        value
    ).resolve()

    try:
        return (
            path
            .relative_to(storage_root)
            .as_posix()
        )

    except ValueError as error:
        raise ValueError(
            "Manifest path is outside storage_root: "
            f"{path}"
        ) from error


# ------------------------------------------------------------------
# Labelled-frame manifest
# ------------------------------------------------------------------

def build_labelled_frame_manifest(
    dataframe,
    frame_index_offset,
    storage_root,
    bbox_spec,
):
    """
    Builds the trusted labelled-frame manifest.

    One row represents one medically verified
    Kvasir-Capsule frame.

    Returns a new DataFrame.
    """

    base_columns = [
        # Image identity and location
        "filename",
        "image_key",
        "image_path",
        "image_exists",

        # Video identity and location
        "video_id",
        "video_key",
        "video_path",
        "video_exists",

        # Temporal ground truth
        "frame_number",

        # Medical ground truth
        "finding_class_raw",
        "finding_class",
        "finding_class_normalized",
        "clinical_group",

        # Bounding-box state
        "bbox_annotation_present",
        "bbox_complete",
        "bbox_invalid",
        "has_bbox",

        # Standardized bounding box
        "bbox_xmin",
        "bbox_ymin",
        "bbox_xmax",
        "bbox_ymax",
        "bbox_width",
        "bbox_height",
        "bbox_area",
    ]

    optional_columns = [
        column
        for column in (
            "finding_category",
            *bbox_spec["x_columns"],
            *bbox_spec["y_columns"],
        )
        if column in dataframe.columns
    ]

    selected_columns = (
        base_columns
        + optional_columns
    )


    # --------------------------------------------------------------
    # Validate required columns
    # --------------------------------------------------------------

    missing_columns = sorted(
        set(base_columns)
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Labelled-frame manifest is missing columns: "
            f"{missing_columns}"
        )


    # --------------------------------------------------------------
    # Validate physical file mappings
    # --------------------------------------------------------------

    unresolved_images = (
        ~dataframe["image_exists"]
        .fillna(False)
    )

    unresolved_videos = (
        ~dataframe["video_exists"]
        .fillna(False)
    )

    if unresolved_images.any():
        raise ValueError(
            "Cannot build trusted manifest: "
            f"{int(unresolved_images.sum())} "
            "labelled images are unresolved."
        )

    if unresolved_videos.any():
        raise ValueError(
            "Cannot build trusted manifest: "
            f"{int(unresolved_videos.sum())} "
            "source videos are unresolved."
        )


    # --------------------------------------------------------------
    # Validate frame numbers
    # --------------------------------------------------------------

    frame_numbers = pd.to_numeric(
        dataframe["frame_number"],
        errors="coerce",
    )

    valid_frame_numbers = (
        frame_numbers.notna()
        & frame_numbers.ge(0)
        & frame_numbers.eq(
            np.floor(frame_numbers)
        )
    )

    if not valid_frame_numbers.all():
        raise ValueError(
            "Cannot build trusted manifest: "
            f"{int((~valid_frame_numbers).sum())} "
            "invalid frame numbers were found."
        )

    frame_numbers = (
        frame_numbers
        .astype("Int64")
    )

    frame_index_offset = int(
        frame_index_offset
    )

    opencv_frame_indices = (
        frame_numbers
        + frame_index_offset
    )

    if opencv_frame_indices.lt(0).any():
        raise ValueError(
            "The selected frame-index offset produces "
            "negative OpenCV frame indices."
        )


    # --------------------------------------------------------------
    # Build immutable-by-contract result
    # --------------------------------------------------------------

    result = (
        dataframe[
            selected_columns
        ]
        .copy()
        .assign(
            frame_number=frame_numbers,

            frame_index_offset=(
                frame_index_offset
            ),

            opencv_frame_index=(
                opencv_frame_indices
            ),

            image_relpath=(
                dataframe["image_path"]
                .map(
                    lambda value:
                    path_relative_to_storage(
                        value=value,
                        storage_root=storage_root,
                    )
                )
            ),

            video_relpath=(
                dataframe["video_path"]
                .map(
                    lambda value:
                    path_relative_to_storage(
                        value=value,
                        storage_root=storage_root,
                    )
                )
            ),

            annotation_status=(
                "medically_verified"
            ),

            label_source=(
                "kvasir_capsule_expert_annotation"
            ),
        )
    )

    return result


labelled_frame_manifest = (
    build_labelled_frame_manifest(
        dataframe=df,
        frame_index_offset=(
            FRAME_INDEX_OFFSET
        ),
        storage_root=CONFIG[
            "storage_root"
        ],
        bbox_spec=BBOX_SPEC,
    )
)


print(
    "Labelled-frame manifest shape:",
    labelled_frame_manifest.shape,
)

display(
    labelled_frame_manifest.head()
)

In [ ]:
LABELLED_FRAME_BOUNDS_REPORT_COLUMNS = [
    "video_key",
    "labelled_frame_count",
    "minimum_opencv_frame_index",
    "maximum_opencv_frame_index",
    "video_frame_count",
    "out_of_bounds_frame_count",
    "validation_passed",
]


def build_labelled_frame_bounds_report(
    labelled_manifest,
    video_manifest,
):
    """
    Verifies that every medically labelled frame index
    falls inside its source video's valid frame range.

    The input DataFrames are not modified.
    """

    required_labelled_columns = {
        "video_key",
        "opencv_frame_index",
    }

    required_video_columns = {
        "video_key",
        "frame_count",
    }

    missing_labelled_columns = sorted(
        required_labelled_columns
        - set(labelled_manifest.columns)
    )

    missing_video_columns = sorted(
        required_video_columns
        - set(video_manifest.columns)
    )

    if missing_labelled_columns:
        raise KeyError(
            "Labelled-frame bounds validation is missing "
            f"columns: {missing_labelled_columns}"
        )

    if missing_video_columns:
        raise KeyError(
            "Video manifest is missing frame-bound columns: "
            f"{missing_video_columns}"
        )

    if not video_manifest[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Video manifest contains duplicate video keys."
        )

    frame_references = (
        labelled_manifest[
            [
                "video_key",
                "opencv_frame_index",
            ]
        ]
        .drop_duplicates()
        .assign(
            opencv_frame_index=lambda data:
                pd.to_numeric(
                    data[
                        "opencv_frame_index"
                    ],
                    errors="coerce",
                )
        )
    )

    invalid_frame_index_mask = (
        frame_references[
            "opencv_frame_index"
        ].isna()
        |
        ~np.isfinite(
            frame_references[
                "opencv_frame_index"
            ].astype("float64")
        )
        |
        frame_references[
            "opencv_frame_index"
        ].mod(1).ne(0)
    )

    if invalid_frame_index_mask.any():
        invalid_frames = (
            frame_references.loc[
                invalid_frame_index_mask
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Labelled manifest contains invalid OpenCV "
            f"frame indices: {invalid_frames}"
        )

    video_frame_counts = (
        video_manifest[
            [
                "video_key",
                "frame_count",
            ]
        ]
        .assign(
            frame_count=lambda data:
                pd.to_numeric(
                    data[
                        "frame_count"
                    ],
                    errors="coerce",
                )
        )
    )

    invalid_frame_count_mask = (
        video_frame_counts[
            "frame_count"
        ].isna()
        |
        ~np.isfinite(
            video_frame_counts[
                "frame_count"
            ].astype("float64")
        )
        |
        video_frame_counts[
            "frame_count"
        ].le(0)
        |
        video_frame_counts[
            "frame_count"
        ].mod(1).ne(0)
    )

    if invalid_frame_count_mask.any():
        invalid_videos = (
            video_frame_counts.loc[
                invalid_frame_count_mask,
                "video_key",
            ]
            .tolist()
        )

        raise ValueError(
            "Invalid video frame counts found for: "
            f"{invalid_videos}"
        )

    checked_frames = (
        frame_references
        .assign(
            opencv_frame_index=lambda data:
                data[
                    "opencv_frame_index"
                ].astype("Int64")
        )
        .merge(
            video_frame_counts.assign(
                frame_count=lambda data:
                    data[
                        "frame_count"
                    ].astype("Int64")
            ),
            on="video_key",
            how="left",
            validate="many_to_one",
            indicator=True,
        )
    )

    missing_video_mask = (
        checked_frames[
            "_merge"
        ].ne("both")
    )

    if missing_video_mask.any():
        missing_video_keys = (
            checked_frames.loc[
                missing_video_mask,
                "video_key",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise KeyError(
            "Labelled frames reference videos missing from "
            f"the video manifest: {missing_video_keys}"
        )

    checked_frames = (
        checked_frames
        .drop(
            columns="_merge"
        )
        .assign(
            frame_out_of_bounds=lambda data:
                (
                    data[
                        "opencv_frame_index"
                    ].lt(0)
                    |
                    data[
                        "opencv_frame_index"
                    ].ge(
                        data[
                            "frame_count"
                        ]
                    )
                )
        )
    )

    return (
        checked_frames
        .groupby(
            "video_key",
            as_index=False,
            sort=True,
            observed=True,
        )
        .agg(
            labelled_frame_count=(
                "opencv_frame_index",
                "size",
            ),

            minimum_opencv_frame_index=(
                "opencv_frame_index",
                "min",
            ),

            maximum_opencv_frame_index=(
                "opencv_frame_index",
                "max",
            ),

            video_frame_count=(
                "frame_count",
                "first",
            ),

            out_of_bounds_frame_count=(
                "frame_out_of_bounds",
                "sum",
            ),
        )
        .assign(
            validation_passed=lambda data:
                data[
                    "out_of_bounds_frame_count"
                ].eq(0)
        )
        .loc[
            :,
            LABELLED_FRAME_BOUNDS_REPORT_COLUMNS,
        ]
    )


def validate_labelled_frame_bounds(
    bounds_report,
):
    """
    Stops the pipeline when labelled frame indices fall
    outside their source video.
    """

    failed_videos = (
        bounds_report.loc[
            ~bounds_report[
                "validation_passed"
            ]
        ]
    )

    if not failed_videos.empty:
        raise ValueError(
            "Medically labelled frames outside video bounds "
            "were detected: "
            f"{failed_videos.to_dict(orient='records')}"
        )

    return bounds_report


labelled_frame_bounds_report = (
    build_labelled_frame_bounds_report(
        labelled_manifest=(
            labelled_frame_manifest
        ),
        video_manifest=video_manifest,
    )
)


display(
    labelled_frame_bounds_report
)


labelled_frame_bounds_report = (
    labelled_frame_bounds_report
    .pipe(
        validate_labelled_frame_bounds
    )
)


print(
    "PASS: all medically labelled frames are within "
    "their source-video bounds."
)

In [ ]:
FRAME_INVENTORY_REPORT_COLUMNS = [
    "frame_category",
    "actual_frame_count",
    "expected_frame_count",
    "validation_passed",
]


def build_frame_inventory_report(
    video_manifest,
    labelled_manifest,
    config,
):
    """
    Builds and validates the global physical-frame inventory.

    A medically verified frame is counted once using:
        video_key + opencv_frame_index

    Unlabelled frames are derived as:
        total extractable frames - unique verified frames
    """

    required_video_columns = {
        "video_key",
        "frame_count",
    }

    required_labelled_columns = {
        "video_key",
        "opencv_frame_index",
    }

    missing_video_columns = sorted(
        required_video_columns
        - set(video_manifest.columns)
    )

    missing_labelled_columns = sorted(
        required_labelled_columns
        - set(labelled_manifest.columns)
    )

    if missing_video_columns:
        raise KeyError(
            "Frame inventory is missing video columns: "
            f"{missing_video_columns}"
        )

    if missing_labelled_columns:
        raise KeyError(
            "Frame inventory is missing labelled-frame "
            f"columns: {missing_labelled_columns}"
        )

    if not video_manifest[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Video manifest contains duplicate video keys."
        )

    video_frame_counts = pd.to_numeric(
        video_manifest[
            "frame_count"
        ],
        errors="coerce",
    )

    invalid_video_frame_count_mask = (
        video_frame_counts.isna()
        | ~np.isfinite(
            video_frame_counts.astype("float64")
        )
        | video_frame_counts.le(0)
        | video_frame_counts.mod(1).ne(0)
    )

    if invalid_video_frame_count_mask.any():
        invalid_video_keys = (
            video_manifest.loc[
                invalid_video_frame_count_mask,
                "video_key",
            ]
            .tolist()
        )

        raise ValueError(
            "Invalid frame counts found for videos: "
            f"{invalid_video_keys}"
        )

    labelled_frame_references = (
        labelled_manifest[
            [
                "video_key",
                "opencv_frame_index",
            ]
        ]
        .drop_duplicates()
        .assign(
            opencv_frame_index=lambda data:
                pd.to_numeric(
                    data[
                        "opencv_frame_index"
                    ],
                    errors="coerce",
                )
        )
    )

    invalid_labelled_index_mask = (
        labelled_frame_references[
            "opencv_frame_index"
        ].isna()
        |
        ~np.isfinite(
            labelled_frame_references[
                "opencv_frame_index"
            ].astype("float64")
        )
        |
        labelled_frame_references[
            "opencv_frame_index"
        ].lt(0)
        |
        labelled_frame_references[
            "opencv_frame_index"
        ].mod(1).ne(0)
    )

    if invalid_labelled_index_mask.any():
        raise ValueError(
            "Labelled-frame inventory contains invalid "
            "OpenCV frame indices."
        )

    physical_video_keys = set(
        video_manifest[
            "video_key"
        ]
    )

    labelled_video_keys = set(
        labelled_frame_references[
            "video_key"
        ]
    )

    unknown_video_keys = sorted(
        labelled_video_keys
        - physical_video_keys
    )

    if unknown_video_keys:
        raise KeyError(
            "Labelled frames reference unknown videos: "
            f"{unknown_video_keys}"
        )

    total_extractable_frames = int(
        video_frame_counts.sum()
    )

    verified_labelled_frames = int(
        len(
            labelled_frame_references
        )
    )

    unlabelled_frames = (
        total_extractable_frames
        - verified_labelled_frames
    )

    if unlabelled_frames < 0:
        raise RuntimeError(
            "Verified frame count exceeds the total "
            "extractable video-frame count."
        )

    expected_total_frames = int(
        config[
            "expected_total_extractable_frames"
        ]
    )

    expected_labelled_frames = int(
        config[
            "expected_labelled_frames"
        ]
    )

    expected_unlabelled_frames = int(
        config[
            "expected_unlabelled_frames"
        ]
    )

    if (
        expected_labelled_frames
        + expected_unlabelled_frames
        != expected_total_frames
    ):
        raise ValueError(
            "CONFIG contains an inconsistent expected "
            "frame inventory: labelled + unlabelled "
            "does not equal total."
        )

    return (
        pd.DataFrame(
            {
                "frame_category": [
                    "total_extractable_frames",
                    "verified_labelled_frames",
                    "unlabelled_frames",
                ],

                "actual_frame_count": [
                    total_extractable_frames,
                    verified_labelled_frames,
                    unlabelled_frames,
                ],

                "expected_frame_count": [
                    expected_total_frames,
                    expected_labelled_frames,
                    expected_unlabelled_frames,
                ],
            }
        )
        .assign(
            validation_passed=lambda data:
                data[
                    "actual_frame_count"
                ].eq(
                    data[
                        "expected_frame_count"
                    ]
                )
        )
        .loc[
            :,
            FRAME_INVENTORY_REPORT_COLUMNS,
        ]
    )


def validate_frame_inventory(
    frame_inventory_report,
):
    """
    Stops the pipeline when the physical-frame inventory
    differs from the expected dataset characteristics.
    """

    failed_checks = (
        frame_inventory_report.loc[
            ~frame_inventory_report[
                "validation_passed"
            ]
        ]
    )

    if not failed_checks.empty:
        raise ValueError(
            "Frame inventory validation failed: "
            f"{failed_checks.to_dict(orient='records')}"
        )

    return frame_inventory_report


frame_inventory_report = (
    build_frame_inventory_report(
        video_manifest=video_manifest,
        labelled_manifest=(
            labelled_frame_manifest
        ),
        config=CONFIG,
    )
)


display(
    frame_inventory_report
)


frame_inventory_report = (
    frame_inventory_report
    .pipe(
        validate_frame_inventory
    )
)


print(
    "PASS: total, labelled, and unlabelled frame "
    "inventories were validated."
)

### 14. Create video/class matrix

In [ ]:
def build_video_class_matrix(
    labelled_manifest,
    expected_classes,
):
    """
    Builds a stable video-level multilabel presence matrix.

    Rows:
        Labelled videos.

    Columns:
        Normalized Kvasir finding classes.

    Values:
        1 when at least one verified frame of the class
        appears in the video; otherwise 0.
    """

    required_columns = {
        "video_key",
        "finding_class_normalized",
    }

    missing_columns = sorted(
        required_columns
        - set(labelled_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot build video-class matrix. "
            f"Missing columns: {missing_columns}"
        )

    video_keys = (
        labelled_manifest["video_key"]
        .astype("string")
        .str.strip()
    )

    finding_classes = (
        labelled_manifest[
            "finding_class_normalized"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_rows = (
        video_keys.isna()
        | video_keys.eq("")
        | finding_classes.isna()
        | finding_classes.eq("")
    )

    if invalid_rows.any():
        raise ValueError(
            "Cannot build video-class matrix: "
            f"{int(invalid_rows.sum())} rows contain "
            "missing video keys or finding classes."
        )

    expected_classes = tuple(
        sorted(
            expected_classes
        )
    )

    observed_classes = set(
        finding_classes.unique()
    )

    missing_classes = sorted(
        set(expected_classes)
        - observed_classes
    )

    unexpected_classes = sorted(
        observed_classes
        - set(expected_classes)
    )

    if missing_classes or unexpected_classes:
        raise ValueError(
            "Video-class taxonomy mismatch. "
            f"Missing classes: {missing_classes}. "
            f"Unexpected classes: {unexpected_classes}."
        )

    matrix = pd.crosstab(
        video_keys,
        finding_classes,
    )

    return (
        matrix
        .gt(0)
        .astype("int8")
        .reindex(
            columns=expected_classes,
            fill_value=0,
        )
        .rename_axis(
            index="video_key",
            columns="finding_class_normalized",
        )
    )

video_class_matrix = (
    build_video_class_matrix(
        labelled_manifest=(
            labelled_frame_manifest
        ),
        expected_classes=(
            CONFIG[
                "clinical_group_map"
            ].keys()
        ),
    )
)


video_class_support = (
    video_class_matrix
    .sum(axis=0)
    .sort_values()
    .rename(
        "labelled_video_count"
    )
    .to_frame()
)


print(
    "Video-class matrix shape:",
    video_class_matrix.shape,
)

display(
    video_class_matrix.head()
)

display(
    video_class_support
)

### 15. Multilabel-stratified video split

In [ ]:
def validate_split_configuration(
    config,
):
    """
    Validates and normalizes the configured video-level split.

    Returns normalized split fractions and random seed.
    """

    required_keys = {
        "train_fraction",
        "validation_fraction",
        "test_fraction",
        "split_strategy",
        "seed",
    }

    missing_keys = sorted(
        required_keys
        - set(config)
    )

    if missing_keys:
        raise KeyError(
            "Split configuration is missing keys: "
            f"{missing_keys}"
        )

    fractions = {
        "train": float(
            config["train_fraction"]
        ),
        "validation": float(
            config["validation_fraction"]
        ),
        "test": float(
            config["test_fraction"]
        ),
    }

    invalid_fractions = {
        name: value
        for name, value in fractions.items()
        if not 0.0 < value < 1.0
    }

    if invalid_fractions:
        raise ValueError(
            "Every split fraction must be between "
            f"0 and 1: {invalid_fractions}"
        )

    if not np.isclose(
        sum(fractions.values()),
        1.0,
    ):
        raise ValueError(
            "train_fraction + validation_fraction + "
            "test_fraction must equal 1.0."
        )

    expected_strategy = (
        "video_level_multilabel_stratified"
    )

    if (
        config["split_strategy"]
        != expected_strategy
    ):
        raise ValueError(
            "Unsupported split strategy: "
            f"{config['split_strategy']}"
        )

    try:
        seed = int(
            config["seed"]
        )

    except (TypeError, ValueError) as error:
        raise ValueError(
            "The split seed must be an integer."
        ) from error

    return {
        **fractions,
        "seed": seed,
    }


def split_labelled_videos(
    video_class_matrix,
    config,
):
    """
    Splits labelled videos using multilabel stratification.

    Returns sets of video keys for train, validation, and test.
    """

    split_config = (
        validate_split_configuration(
            config
        )
    )

    if video_class_matrix.empty:
        raise ValueError(
            "Cannot split an empty video-class matrix."
        )

    if not video_class_matrix.index.is_unique:
        raise ValueError(
            "Video-class matrix contains duplicate video keys."
        )

    if video_class_matrix.isna().any().any():
        raise ValueError(
            "Video-class matrix contains missing label values."
        )

    valid_label_values = (
        video_class_matrix
        .isin(
            [0, 1]
        )
        .all()
        .all()
    )

    if not valid_label_values:
        raise ValueError(
            "Video-class matrix must contain only binary "
            "label values: 0 or 1."
        )

    video_keys = (
        video_class_matrix
        .index
        .to_numpy()
    )

    labels = (
        video_class_matrix
        .to_numpy(
            dtype=np.int8
        )
    )

    dummy_features = np.zeros(
        (
            len(video_keys),
            1,
        ),
        dtype=np.int8,
    )

    temporary_fraction = (
        split_config["validation"]
        + split_config["test"]
    )

    first_splitter = (
        MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=temporary_fraction,
            random_state=split_config["seed"],
        )
    )

    train_indices, temporary_indices = next(
        first_splitter.split(
            dummy_features,
            labels,
        )
    )

    relative_test_fraction = (
        split_config["test"]
        / temporary_fraction
    )

    second_splitter = (
        MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=relative_test_fraction,
            random_state=split_config["seed"],
        )
    )

    validation_relative, test_relative = next(
        second_splitter.split(
            dummy_features[
                temporary_indices
            ],
            labels[
                temporary_indices
            ],
        )
    )

    validation_indices = (
        temporary_indices[
            validation_relative
        ]
    )

    test_indices = (
        temporary_indices[
            test_relative
        ]
    )

    return {
        "train": set(
            video_keys[
                train_indices
            ]
        ),
        "validation": set(
            video_keys[
                validation_indices
            ]
        ),
        "test": set(
            video_keys[
                test_indices
            ]
        ),
    }


VIDEO_SPLITS = split_labelled_videos(
    video_class_matrix=video_class_matrix,
    config=CONFIG,
)

VIDEO_SPLITS

### 16. Propagate split



In [ ]:
def validate_video_splits(
    video_splits,
    expected_video_keys,
):
    """
    Validates that labelled videos form a complete,
    non-overlapping train/validation/test partition.
    """

    required_split_names = {
        "train",
        "validation",
        "test",
    }

    actual_split_names = set(
        video_splits
    )

    missing_split_names = sorted(
        required_split_names
        - actual_split_names
    )

    unexpected_split_names = sorted(
        actual_split_names
        - required_split_names
    )

    if (
        missing_split_names
        or unexpected_split_names
    ):
        raise KeyError(
            "Invalid video split names. "
            f"Missing: {missing_split_names}. "
            f"Unexpected: {unexpected_split_names}."
        )

    train = set(
        video_splits["train"]
    )

    validation = set(
        video_splits["validation"]
    )

    test = set(
        video_splits["test"]
    )

    empty_splits = [
        split_name
        for split_name, video_keys
        in {
            "train": train,
            "validation": validation,
            "test": test,
        }.items()
        if not video_keys
    ]

    if empty_splits:
        raise RuntimeError(
            "Empty video splits found: "
            f"{empty_splits}"
        )

    overlaps = {
        "train_validation": (
            train
            & validation
        ),
        "train_test": (
            train
            & test
        ),
        "validation_test": (
            validation
            & test
        ),
    }

    nonempty_overlaps = {
        name: sorted(video_keys)
        for name, video_keys
        in overlaps.items()
        if video_keys
    }

    if nonempty_overlaps:
        raise RuntimeError(
            "Video leakage detected between splits: "
            f"{nonempty_overlaps}"
        )

    expected_video_keys = set(
        expected_video_keys
    )

    assigned_video_keys = (
        train
        | validation
        | test
    )

    missing_video_keys = sorted(
        expected_video_keys
        - assigned_video_keys
    )

    unexpected_video_keys = sorted(
        assigned_video_keys
        - expected_video_keys
    )

    if (
        missing_video_keys
        or unexpected_video_keys
    ):
        raise RuntimeError(
            "Video split coverage mismatch. "
            f"Missing videos: {missing_video_keys}. "
            f"Unexpected videos: {unexpected_video_keys}."
        )


def build_split_lookup(
    video_splits,
):
    """
    Creates a video_key-to-split mapping.
    """

    return {
        video_key: split_name
        for split_name, video_keys
        in video_splits.items()
        for video_key in video_keys
    }


validate_video_splits(
    video_splits=VIDEO_SPLITS,
    expected_video_keys=(
        video_class_matrix.index
    ),
)


SPLIT_LOOKUP = build_split_lookup(
    VIDEO_SPLITS
)


labelled_frame_manifest = (
    labelled_frame_manifest
    .assign(
        split=lambda data:
            data["video_key"]
            .map(SPLIT_LOOKUP)
    )
)


unassigned_frame_count = int(
    labelled_frame_manifest[
        "split"
    ]
    .isna()
    .sum()
)

if unassigned_frame_count:
    raise RuntimeError(
        "Some labelled frames did not receive a split: "
        f"{unassigned_frame_count}"
    )

### 17. Assign roles to all videos

In [ ]:
def derive_video_role(
    annotation_type,
    supervised_split,
    config,
):
    """
    Determines the validated Phase 2 role of one video.

    The global domain-adaptation switch takes precedence
    over the individual source-inclusion switches.
    """

    required_config_keys = {
        "domain_adaptation_enabled",
        "domain_adaptation_include_fully_unlabelled_videos",
        "domain_adaptation_include_train_video_unlabelled_frames",
    }

    missing_config_keys = sorted(
        required_config_keys
        - set(config)
    )

    if missing_config_keys:
        raise KeyError(
            "Domain-adaptation configuration is missing "
            f"keys: {missing_config_keys}"
        )

    if pd.isna(annotation_type):
        raise ValueError(
            "Video annotation type is missing."
        )

    annotation_type = str(
        annotation_type
    )

    has_supervised_split = (
        pd.notna(
            supervised_split
        )
    )

    domain_adaptation_enabled = bool(
        config[
            "domain_adaptation_enabled"
        ]
    )

    include_fully_unlabelled = (
        domain_adaptation_enabled
        and bool(
            config[
                "domain_adaptation_include_fully_unlabelled_videos"
            ]
        )
    )

    include_train_unlabelled = (
        domain_adaptation_enabled
        and bool(
            config[
                "domain_adaptation_include_train_video_unlabelled_frames"
            ]
        )
    )

    if annotation_type == "fully_unlabelled":

        if has_supervised_split:
            raise ValueError(
                "A fully unlabelled video cannot belong "
                "to a supervised split."
            )

        return (
            "domain_adaptation_only"
            if include_fully_unlabelled
            else "unlabelled_reserved"
        )

    if annotation_type != "partially_labelled":
        raise ValueError(
            "Unsupported video annotation type: "
            f"{annotation_type}"
        )

    if not has_supervised_split:
        raise ValueError(
            "A partially labelled video did not receive "
            "a supervised split."
        )

    if supervised_split == "train":
        return (
            "supervised_train_plus_domain_adaptation"
            if include_train_unlabelled
            else "supervised_train"
        )

    if supervised_split == "validation":
        return "supervised_validation"

    if supervised_split == "test":
        return "supervised_test"

    raise ValueError(
        "Unsupported supervised split: "
        f"{supervised_split}"
    )


def assign_video_roles(
    video_manifest,
    split_lookup,
    config,
    storage_root,
):
    """
    Adds the supervised split and validated Phase 2
    data role to every video.

    Returns a new DataFrame.
    """

    required_columns = {
        "video_key",
        "video_path",
        "video_annotation_type",
    }

    missing_columns = sorted(
        required_columns
        - set(video_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot assign video roles. "
            f"Missing columns: {missing_columns}"
        )

    if not video_manifest[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Video manifest contains duplicate video keys."
        )

    result = (
        video_manifest
        .copy()
        .assign(
            supervised_split=lambda data:
                data["video_key"]
                .map(split_lookup),

            video_relpath=lambda data:
                data["video_path"]
                .map(
                    lambda value:
                        path_relative_to_storage(
                            value=value,
                            storage_root=storage_root,
                        )
                ),
        )
    )

    result["data_role"] = [
        derive_video_role(
            annotation_type=annotation_type,
            supervised_split=supervised_split,
            config=config,
        )
        for annotation_type, supervised_split
        in zip(
            result["video_annotation_type"],
            result["supervised_split"],
        )
    ]

    return result


video_manifest = assign_video_roles(
    video_manifest=video_manifest,
    split_lookup=SPLIT_LOOKUP,
    config=CONFIG,
    storage_root=CONFIG[
        "storage_root"
    ],
)


video_role_report = (
    video_manifest[
        [
            "video_annotation_type",
            "supervised_split",
            "data_role",
        ]
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "video_count"
    )
    .reset_index()
)


display(
    video_role_report
)

### 18. Identify domain-adaptation source videos

In [ ]:
def build_verified_frame_lookup(
    labelled_manifest,
):
    """
    Returns verified OpenCV frame indices grouped by video.

    Used to prevent medically labelled frames from entering
    the unlabelled domain-adaptation pool.
    """

    return (
        labelled_manifest

        .groupby(
            "video_key"
        )[
            "opencv_frame_index"
        ]

        .agg(
            lambda values:
                frozenset(
                    values
                    .dropna()
                    .astype(int)
                )
        )

        .to_dict()
    )


def select_domain_adaptation_videos(
    video_manifest,
    config,
):
    """
    Selects videos whose validated data roles allow them
    to contribute unlabelled domain-adaptation frames.

    Returns an empty table with the same schema when
    domain adaptation is disabled.
    """

    if (
        "domain_adaptation_enabled"
        not in config
    ):
        raise KeyError(
            "CONFIG is missing "
            "domain_adaptation_enabled."
        )

    if not bool(
        config[
            "domain_adaptation_enabled"
        ]
    ):
        return (
            video_manifest
            .iloc[
                0:0
            ]
            .copy()
        )

    required_columns = {
        "video_key",
        "data_role",
    }

    missing_columns = sorted(
        required_columns
        - set(video_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot select domain-adaptation videos. "
            f"Missing columns: {missing_columns}"
        )

    allowed_roles = {
        "domain_adaptation_only",
        "supervised_train_plus_domain_adaptation",
    }

    return (
        video_manifest.loc[
            video_manifest[
                "data_role"
            ].isin(
                allowed_roles
            )
        ]
        .copy()
        .sort_values(
            "video_key",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

VERIFIED_FRAME_LOOKUP = (
    build_verified_frame_lookup(
        labelled_frame_manifest
    )
)


domain_source_videos = (
    select_domain_adaptation_videos(
        video_manifest=(
            video_manifest
        ),
        config=CONFIG,
    )
)


print(
    "Domain-adaptation source videos:",
    len(domain_source_videos),
)

### 19. Generate domain-adaptation references

In [ ]:
def get_domain_sampling_stride(
    config,
    video_fps,
):
    """
    Converts the configured sampling interval in seconds
    into frame units for one source video.
    """

    sampling_seconds = float(
        config[
            "domain_adaptation_sampling_seconds"
        ]
    )

    video_fps = float(
        video_fps
    )

    if (
        not np.isfinite(sampling_seconds)
        or sampling_seconds <= 0
    ):
        raise ValueError(
            "Domain-adaptation sampling seconds must "
            "be a positive finite number."
        )

    if (
        not np.isfinite(video_fps)
        or video_fps <= 0
    ):
        raise ValueError(
            "Video FPS must be a positive finite number."
        )

    return max(
        1,
        int(
            round(
                sampling_seconds
                * video_fps
            )
        ),
    )


def build_domain_records_for_video(
    video,
    verified_frame_lookup,
    config,
    frame_index_offset,
):
    """
    Builds unlabelled frame references for one eligible video.

    verified_frame_lookup must contain OpenCV frame indices.

    No images are extracted.
    No pseudo-labels are created.
    """

    required_fields = {
        "video_key",
        "video_relpath",
        "video_annotation_type",
        "supervised_split",
        "data_role",
        "frame_count",
        "container_fps",
    }

    missing_fields = sorted(
        required_fields
        - set(video.index)
    )

    if missing_fields:
        raise KeyError(
            "Cannot build domain-adaptation records. "
            f"Missing video fields: {missing_fields}"
        )

    annotation_type = str(
        video["video_annotation_type"]
    )

    supervised_split = (
        None
        if pd.isna(
            video["supervised_split"]
        )
        else str(
            video["supervised_split"]
        )
    )

    data_role = str(
        video["data_role"]
    )

    is_fully_unlabelled_source = (
        annotation_type
        == "fully_unlabelled"
        and supervised_split is None
        and data_role
        == "domain_adaptation_only"
    )

    is_train_unlabelled_source = (
        annotation_type
        == "partially_labelled"
        and supervised_split
        == "train"
        and data_role
        == "supervised_train_plus_domain_adaptation"
    )

    if not (
        is_fully_unlabelled_source
        or is_train_unlabelled_source
    ):
        raise ValueError(
            "Video is not eligible for domain adaptation: "
            f"{video['video_key']}"
        )

    frame_count_value = float(
        video["frame_count"]
    )

    if (
        not np.isfinite(frame_count_value)
        or frame_count_value <= 0
        or not frame_count_value.is_integer()
    ):
        raise ValueError(
            "Invalid frame count for video: "
            f"{video['video_key']}"
        )

    frame_count = int(
        frame_count_value
    )

    video_fps = float(
        video["container_fps"]
    )

    stride = get_domain_sampling_stride(
        config=config,
        video_fps=video_fps,
    )

    opencv_indices = np.arange(
        0,
        frame_count,
        stride,
        dtype=np.int64,
    )

    verified_indices = np.asarray(
        tuple(
            verified_frame_lookup.get(
                video["video_key"],
                frozenset(),
            )
        ),
        dtype=np.int64,
    )

    if verified_indices.size:

        opencv_indices = opencv_indices[
            ~np.isin(
                opencv_indices,
                verified_indices,
            )
        ]

    frame_index_offset = int(
        frame_index_offset
    )

    metadata_frame_numbers = (
        opencv_indices
        - frame_index_offset
    )

    valid_mask = (
        metadata_frame_numbers
        >= 0
    )

    opencv_indices = opencv_indices[
        valid_mask
    ]

    metadata_frame_numbers = (
        metadata_frame_numbers[
            valid_mask
        ]
    )

    source_type = (
        "fully_unlabelled_video"
        if is_fully_unlabelled_source
        else "unlabelled_frame_from_train_video"
    )

    return pd.DataFrame(
        {
            "video_key":
                video["video_key"],

            "video_relpath":
                video["video_relpath"],

            "opencv_frame_index":
                opencv_indices,

            "frame_number":
                metadata_frame_numbers,

            "source_timestamp_seconds":
                (
                    opencv_indices
                    / video_fps
                ),

            "source_type":
                source_type,

            "annotation_status":
                "unlabelled",

            "ground_truth_label":
                pd.NA,

            "supervised_split":
                (
                    supervised_split
                    if supervised_split is not None
                    else pd.NA
                ),

            "purpose":
                "visual_domain_adaptation",
        }
    )

### 20. Build domain-adaptation manifest

In [ ]:
DOMAIN_ADAPTATION_COLUMNS = [
    "video_key",
    "video_relpath",
    "opencv_frame_index",
    "frame_number",
    "source_timestamp_seconds",
    "source_type",
    "annotation_status",
    "ground_truth_label",
    "supervised_split",
    "purpose",
]


def build_domain_adaptation_manifest(
    domain_source_videos,
    verified_frame_lookup,
    config,
    frame_index_offset,
):
    """
    Builds the complete domain-adaptation reference manifest.

    One row represents one unlabelled frame reference.
    No images are extracted or saved.
    """

    required_columns = {
        "video_key",
        "video_relpath",
        "video_annotation_type",
        "supervised_split",
        "data_role",
        "frame_count",
        "container_fps",
    }

    missing_columns = sorted(
        required_columns
        - set(domain_source_videos.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot build domain-adaptation manifest. "
            f"Missing columns: {missing_columns}"
        )

    if not domain_source_videos[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Domain source videos contain duplicate "
            "video keys."
        )

    eligible_roles = {
        "domain_adaptation_only",
        "supervised_train_plus_domain_adaptation",
    }

    invalid_role_mask = (
        ~domain_source_videos[
            "data_role"
        ]
        .isin(
            eligible_roles
        )
    )

    if invalid_role_mask.any():

        invalid_videos = (
            domain_source_videos.loc[
                invalid_role_mask,
                [
                    "video_key",
                    "supervised_split",
                    "data_role",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Non-eligible videos were included in the "
            "domain-adaptation source pool: "
            f"{invalid_videos}"
        )

    if domain_source_videos.empty:
        return pd.DataFrame(
            columns=DOMAIN_ADAPTATION_COLUMNS
        )

    domain_tables = [
        build_domain_records_for_video(
            video=video,
            verified_frame_lookup=(
                verified_frame_lookup
            ),
            config=config,
            frame_index_offset=(
                frame_index_offset
            ),
        )
        for _, video
        in tqdm(
            domain_source_videos.iterrows(),
            total=len(
                domain_source_videos
            ),
            desc=(
                "Building domain-adaptation manifest"
            ),
        )
    ]

    non_empty_tables = [
        table
        for table in domain_tables
        if not table.empty
    ]

    if not non_empty_tables:
        return pd.DataFrame(
            columns=DOMAIN_ADAPTATION_COLUMNS
        )

    result = (
        pd.concat(
            non_empty_tables,
            ignore_index=True,
        )
        .loc[
            :,
            DOMAIN_ADAPTATION_COLUMNS,
        ]
        .sort_values(
            [
                "video_key",
                "opencv_frame_index",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    duplicate_reference_mask = (
        result.duplicated(
            subset=[
                "video_key",
                "opencv_frame_index",
            ],
            keep=False,
        )
    )

    if duplicate_reference_mask.any():

        duplicate_references = (
            result.loc[
                duplicate_reference_mask,
                [
                    "video_key",
                    "opencv_frame_index",
                ],
            ]
            .drop_duplicates()
            .to_dict(
                orient="records"
            )
        )

        raise RuntimeError(
            "Duplicate domain-adaptation frame "
            "references found: "
            f"{duplicate_references}"
        )

    return result


domain_adaptation_manifest = (
    build_domain_adaptation_manifest(
        domain_source_videos=(
            domain_source_videos
        ),
        verified_frame_lookup=(
            VERIFIED_FRAME_LOOKUP
        ),
        config=CONFIG,
        frame_index_offset=(
            FRAME_INDEX_OFFSET
        ),
    )
)


domain_adaptation_summary = (
    domain_adaptation_manifest
    .groupby(
        "source_type",
        dropna=False,
    )
    .size()
    .rename(
        "frame_reference_count"
    )
    .reset_index()
)


print(
    "Domain-adaptation frame references:",
    f"{len(domain_adaptation_manifest):,}",
)

display(
    domain_adaptation_summary
)

display(
    domain_adaptation_manifest.head()
)

### 21. Build verified finding segments

In [ ]:
def add_verified_segment_numbers(
    labelled_manifest,
    config,
):
    """
    Groups temporally adjacent verified frames of the same
    finding within the same video.

    Returns a new DataFrame with a segment_number column.
    """

    group_columns = [
        "video_key",
        "finding_class_normalized",
    ]

    required_columns = {
        *group_columns,
        "frame_number",
    }

    missing_columns = sorted(
        required_columns
        - set(labelled_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot build verified segments. "
            f"Missing columns: {missing_columns}"
        )

    max_gap = config[
        "segment_max_gap_frames"
    ]

    if (
        isinstance(max_gap, bool)
        or not isinstance(
            max_gap,
            (int, np.integer),
        )
        or max_gap < 1
    ):
        raise ValueError(
            "segment_max_gap_frames must be "
            "a positive integer."
        )

    result = (
        labelled_manifest
        .sort_values(
            group_columns
            + ["frame_number"],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    frame_gap = (
        result
        .groupby(
            group_columns,
            sort=False,
            observed=True,
        )[
            "frame_number"
        ]
        .diff()
    )

    starts_new_segment = (
        frame_gap.isna()
        | frame_gap.gt(
            max_gap
        )
    )

    segment_numbers = (
        starts_new_segment
        .astype("int8")
        .groupby(
            [
                result["video_key"],
                result[
                    "finding_class_normalized"
                ],
            ],
            sort=False,
            observed=True,
        )
        .cumsum()
        .astype("Int64")
    )

    return result.assign(
        segment_number=segment_numbers
    )


labelled_frame_manifest = (
    add_verified_segment_numbers(
        labelled_manifest=(
            labelled_frame_manifest
        ),
        config=CONFIG,
    )
)

### 22. Create stable segment IDs

In [ ]:
def normalize_identifier_series(
    values,
):
    """
    Converts a pandas Series into stable identifier
    components using vectorized string operations.
    """

    return (
        values
        .astype("string")
        .str.strip()
        .str.casefold()
        .str.replace(
            r"[^a-z0-9]+",
            "_",
            regex=True,
        )
        .str.strip("_")
    )


def add_finding_segment_ids(
    labelled_manifest,
):
    """
    Creates stable IDs for verified finding segments.

    Every frame belonging to the same verified segment
    receives the same finding_segment_id.
    """

    identity_columns = [
        "video_key",
        "finding_class_normalized",
        "segment_number",
    ]

    missing_columns = sorted(
        set(identity_columns)
        - set(labelled_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot create finding segment IDs. "
            f"Missing columns: {missing_columns}"
        )

    result = (
        labelled_manifest
        .copy()
    )

    video_ids = (
        normalize_identifier_series(
            result["video_key"]
        )
    )

    class_ids = (
        normalize_identifier_series(
            result[
                "finding_class_normalized"
            ]
        )
    )

    segment_numbers = pd.to_numeric(
        result["segment_number"],
        errors="coerce",
    )

    invalid_video_ids = (
        video_ids.isna()
        | video_ids.eq("")
    )

    invalid_class_ids = (
        class_ids.isna()
        | class_ids.eq("")
    )

    invalid_segment_numbers = (
        segment_numbers.isna()
        | segment_numbers.lt(1)
        | segment_numbers.mod(1).ne(0)
    )

    if invalid_video_ids.any():
        raise ValueError(
            "Cannot create finding segment IDs: "
            f"{int(invalid_video_ids.sum())} rows have "
            "invalid video keys."
        )

    if invalid_class_ids.any():
        raise ValueError(
            "Cannot create finding segment IDs: "
            f"{int(invalid_class_ids.sum())} rows have "
            "invalid finding classes."
        )

    if invalid_segment_numbers.any():
        raise ValueError(
            "Cannot create finding segment IDs: "
            f"{int(invalid_segment_numbers.sum())} rows "
            "have invalid segment numbers."
        )

    segment_number_ids = (
        segment_numbers
        .astype("Int64")
        .astype("string")
        .str.zfill(5)
    )

    finding_segment_ids = (
        video_ids
        .str.cat(
            class_ids,
            sep="__",
        )
        .str.cat(
            segment_number_ids,
            sep="__",
        )
    )

    result = result.assign(
        segment_number=(
            segment_numbers
            .astype("Int64")
        ),
        finding_segment_id=(
            finding_segment_ids
        ),
    )

    segment_identity = (
        result[
            identity_columns
            + ["finding_segment_id"]
        ]
        .drop_duplicates()
    )

    collision_mask = (
        segment_identity[
            "finding_segment_id"
        ]
        .duplicated(
            keep=False
        )
    )

    if collision_mask.any():

        collisions = (
            segment_identity.loc[
                collision_mask
            ]
            .sort_values(
                "finding_segment_id"
            )
            .to_dict(
                orient="records"
            )
        )

        raise RuntimeError(
            "Finding segment ID collisions detected: "
            f"{collisions}"
        )

    return result


labelled_frame_manifest = (
    add_finding_segment_ids(
        labelled_manifest=(
            labelled_frame_manifest
        )
    )
)

### 23. Build finding-segment manifest

In [ ]:
SEGMENT_MANIFEST_COLUMNS = [
    "finding_segment_id",
    "video_key",
    "finding_class",
    "finding_class_normalized",
    "clinical_group",
    "verified_start_frame",
    "verified_end_frame",
    "verified_frame_count",
    "target_frame_number",
    "target_opencv_frame_index",
    "target_filename",
    "target_image_path",
    "target_image_relpath",
    "target_has_bbox",
    "target_bbox_xmin",
    "target_bbox_ymin",
    "target_bbox_xmax",
    "target_bbox_ymax",
    "split",
]


def build_finding_segment_manifest(
    labelled_manifest,
):
    """
    Aggregates verified frame-level annotations into
    one record per finding segment.

    The representative target is the middle verified
    frame in temporal order.
    """

    required_columns = {
        "finding_segment_id",
        "video_key",
        "finding_class",
        "finding_class_normalized",
        "clinical_group",
        "frame_number",
        "opencv_frame_index",
        "filename",
        "image_path",
        "image_relpath",
        "has_bbox",
        "bbox_xmin",
        "bbox_ymin",
        "bbox_xmax",
        "bbox_ymax",
        "split",
    }

    missing_columns = sorted(
        required_columns
        - set(labelled_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot build finding segment manifest. "
            f"Missing columns: {missing_columns}"
        )

    if labelled_manifest.empty:
        return pd.DataFrame(
            columns=SEGMENT_MANIFEST_COLUMNS
        )

    invalid_segment_id_mask = (
        labelled_manifest[
            "finding_segment_id"
        ]
        .astype("string")
        .str.strip()
        .isna()
        |
        labelled_manifest[
            "finding_segment_id"
        ]
        .astype("string")
        .str.strip()
        .eq("")
    )

    if invalid_segment_id_mask.any():
        raise ValueError(
            "Finding segment manifest contains "
            f"{int(invalid_segment_id_mask.sum())} "
            "rows without a valid segment ID."
        )

    result = (
        labelled_manifest
        .sort_values(
            [
                "finding_segment_id",
                "frame_number",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    duplicate_frame_mask = (
        result.duplicated(
            subset=[
                "finding_segment_id",
                "frame_number",
            ],
            keep=False,
        )
    )

    if duplicate_frame_mask.any():

        duplicate_frames = (
            result.loc[
                duplicate_frame_mask,
                [
                    "finding_segment_id",
                    "frame_number",
                ],
            ]
            .drop_duplicates()
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Duplicate verified frames found within "
            f"finding segments: {duplicate_frames}"
        )

    invariant_columns = [
        "video_key",
        "finding_class",
        "finding_class_normalized",
        "clinical_group",
        "split",
    ]

    invariant_counts = (
        result
        .groupby(
            "finding_segment_id",
            sort=False,
            observed=True,
        )[
            invariant_columns
        ]
        .nunique(
            dropna=False
        )
    )

    inconsistent_segment_mask = (
        invariant_counts
        .gt(1)
        .any(axis=1)
    )

    if inconsistent_segment_mask.any():

        inconsistent_segment_ids = (
            invariant_counts.index[
                inconsistent_segment_mask
            ]
            .tolist()
        )

        raise ValueError(
            "Inconsistent metadata found within "
            "finding segments: "
            f"{inconsistent_segment_ids}"
        )

    segment_group = (
        result.groupby(
            "finding_segment_id",
            sort=False,
            observed=True,
        )
    )

    segment_positions = (
        segment_group
        .cumcount()
    )

    segment_sizes = (
        segment_group[
            "frame_number"
        ]
        .transform(
            "size"
        )
    )

    target_mask = (
        segment_positions
        .eq(
            segment_sizes
            .floordiv(2)
        )
    )

    segment_summary = (
        segment_group
        .agg(
            video_key=(
                "video_key",
                "first",
            ),
            finding_class=(
                "finding_class",
                "first",
            ),
            finding_class_normalized=(
                "finding_class_normalized",
                "first",
            ),
            clinical_group=(
                "clinical_group",
                "first",
            ),
            verified_start_frame=(
                "frame_number",
                "min",
            ),
            verified_end_frame=(
                "frame_number",
                "max",
            ),
            verified_frame_count=(
                "frame_number",
                "size",
            ),
            split=(
                "split",
                "first",
            ),
        )
        .reset_index()
    )

    target_frames = (
        result.loc[
            target_mask,
            [
                "finding_segment_id",
                "frame_number",
                "opencv_frame_index",
                "filename",
                "image_path",
                "image_relpath",
                "has_bbox",
                "bbox_xmin",
                "bbox_ymin",
                "bbox_xmax",
                "bbox_ymax",
            ],
        ]
        .rename(
            columns={
                "frame_number":
                    "target_frame_number",

                "opencv_frame_index":
                    "target_opencv_frame_index",

                "filename":
                    "target_filename",

                "image_path":
                    "target_image_path",

                "image_relpath":
                    "target_image_relpath",

                "has_bbox":
                    "target_has_bbox",

                "bbox_xmin":
                    "target_bbox_xmin",

                "bbox_ymin":
                    "target_bbox_ymin",

                "bbox_xmax":
                    "target_bbox_xmax",

                "bbox_ymax":
                    "target_bbox_ymax",
            }
        )
    )

    finding_segment_manifest = (
        segment_summary
        .merge(
            target_frames,
            on="finding_segment_id",
            how="left",
            validate="one_to_one",
        )
        .loc[
            :,
            SEGMENT_MANIFEST_COLUMNS,
        ]
        .sort_values(
            [
                "video_key",
                "verified_start_frame",
                "finding_class_normalized",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return finding_segment_manifest


finding_segment_manifest = (
    build_finding_segment_manifest(
        labelled_manifest=(
            labelled_frame_manifest
        )
    )
)


print(
    "Verified finding segments:",
    f"{len(finding_segment_manifest):,}",
)

display(
    finding_segment_manifest.head()
)

In [ ]:
SPLIT_CLASS_DISTRIBUTION_COLUMNS = [
    "split",
    "finding_class_normalized",
    "clinical_group",
    "labelled_frame_count",
    "video_count",
    "segment_count",
    "frame_fraction_within_split",
    "class_present_in_split",
]


def build_split_class_distribution_report(
    labelled_manifest,
    segment_manifest,
):
    """
    Builds a split-level class-distribution report.

    Counts are computed independently for verified frames,
    source videos, and verified finding segments.

    Missing split/class combinations are retained with
    zero counts.
    """

    required_labelled_columns = {
        "split",
        "video_key",
        "opencv_frame_index",
        "finding_class_normalized",
        "clinical_group",
    }

    required_segment_columns = {
        "finding_segment_id",
        "split",
        "video_key",
        "finding_class_normalized",
    }

    missing_labelled_columns = sorted(
        required_labelled_columns
        - set(labelled_manifest.columns)
    )

    missing_segment_columns = sorted(
        required_segment_columns
        - set(segment_manifest.columns)
    )

    if missing_labelled_columns:
        raise KeyError(
            "Split-class report is missing labelled-frame "
            f"columns: {missing_labelled_columns}"
        )

    if missing_segment_columns:
        raise KeyError(
            "Split-class report is missing segment columns: "
            f"{missing_segment_columns}"
        )

    expected_splits = pd.DataFrame(
        {
            "split": [
                "train",
                "validation",
                "test",
            ]
        }
    )

    actual_splits = set(
        labelled_manifest[
            "split"
        ]
        .dropna()
        .astype("string")
        .unique()
    )

    unexpected_splits = sorted(
        actual_splits
        - set(
            expected_splits[
                "split"
            ]
        )
    )

    if unexpected_splits:
        raise ValueError(
            "Unexpected supervised split names: "
            f"{unexpected_splits}"
        )

    taxonomy = (
        labelled_manifest[
            [
                "finding_class_normalized",
                "clinical_group",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "clinical_group",
                "finding_class_normalized",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    inconsistent_taxonomy_mask = (
        taxonomy[
            "finding_class_normalized"
        ]
        .duplicated(
            keep=False
        )
    )

    if inconsistent_taxonomy_mask.any():
        inconsistent_classes = (
            taxonomy.loc[
                inconsistent_taxonomy_mask
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Finding classes map to inconsistent clinical "
            f"groups: {inconsistent_classes}"
        )

    split_class_grid = (
        expected_splits
        .merge(
            taxonomy,
            how="cross",
        )
    )

    frame_support = (
        labelled_manifest[
            [
                "split",
                "finding_class_normalized",
                "video_key",
                "opencv_frame_index",
            ]
        ]
        .drop_duplicates()
        .groupby(
            [
                "split",
                "finding_class_normalized",
            ],
            as_index=False,
            sort=False,
            observed=True,
        )
        .agg(
            labelled_frame_count=(
                "opencv_frame_index",
                "size",
            ),

            video_count=(
                "video_key",
                "nunique",
            ),
        )
    )

    segment_support = (
        segment_manifest[
            [
                "finding_segment_id",
                "split",
                "finding_class_normalized",
            ]
        ]
        .drop_duplicates(
            subset=[
                "finding_segment_id",
            ]
        )
        .groupby(
            [
                "split",
                "finding_class_normalized",
            ],
            as_index=False,
            sort=False,
            observed=True,
        )
        .agg(
            segment_count=(
                "finding_segment_id",
                "nunique",
            )
        )
    )

    count_columns = [
        "labelled_frame_count",
        "video_count",
        "segment_count",
    ]

    result = (
        split_class_grid
        .merge(
            frame_support,
            on=[
                "split",
                "finding_class_normalized",
            ],
            how="left",
            validate="one_to_one",
        )
        .merge(
            segment_support,
            on=[
                "split",
                "finding_class_normalized",
            ],
            how="left",
            validate="one_to_one",
        )
        .assign(
            **{
                column:
                    lambda data, column=column:
                        data[
                            column
                        ]
                        .fillna(0)
                        .astype("Int64")

                for column in count_columns
            }
        )
        .assign(
            frame_fraction_within_split=lambda data:
                (
                    data[
                        "labelled_frame_count"
                    ]
                    .astype("float64")
                    .div(
                        data
                        .groupby(
                            "split",
                            observed=True,
                        )[
                            "labelled_frame_count"
                        ]
                        .transform("sum")
                        .astype("float64")
                    )
                    .fillna(0.0)
                ),

            class_present_in_split=lambda data:
                data[
                    "labelled_frame_count"
                ].gt(0),
        )
        .loc[
            :,
            SPLIT_CLASS_DISTRIBUTION_COLUMNS,
        ]
        .sort_values(
            [
                "split",
                "labelled_frame_count",
                "finding_class_normalized",
            ],
            ascending=[
                True,
                False,
                True,
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return result


split_class_distribution_report = (
    build_split_class_distribution_report(
        labelled_manifest=(
            labelled_frame_manifest
        ),
        segment_manifest=(
            finding_segment_manifest
        ),
    )
)


missing_split_class_support = (
    split_class_distribution_report.loc[
        ~split_class_distribution_report[
            "class_present_in_split"
        ]
    ]
)


print(
    "Split/class combinations:",
    f"{len(split_class_distribution_report):,}",
)


print(
    "Combinations without labelled frames:",
    f"{len(missing_split_class_support):,}",
)


display(
    split_class_distribution_report
)


if not missing_split_class_support.empty:
    display(
        missing_split_class_support
    )

### 24. Determine temporal-grounding supervision

In [ ]:
def add_temporal_supervision_metadata(
    segment_manifest,
    config,
):
    """
    Describes the verified temporal supervision available
    for each finding segment.

    Observed segment limits are not treated as exact clinical
    onset or offset boundaries.
    """

    required_columns = {
        "verified_start_frame",
        "verified_end_frame",
        "verified_frame_count",
    }

    missing_columns = sorted(
        required_columns
        - set(segment_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot add temporal supervision metadata. "
            f"Missing columns: {missing_columns}"
        )

    minimum_frames = config[
        "minimum_verified_segment_frames"
    ]

    if (
        isinstance(minimum_frames, bool)
        or not isinstance(
            minimum_frames,
            (int, np.integer),
        )
        or minimum_frames < 2
    ):
        raise ValueError(
            "minimum_verified_segment_frames must be "
            "an integer greater than or equal to 2."
        )

    verified_frame_counts = pd.to_numeric(
        segment_manifest[
            "verified_frame_count"
        ],
        errors="coerce",
    )

    verified_start_frames = pd.to_numeric(
        segment_manifest[
            "verified_start_frame"
        ],
        errors="coerce",
    )

    verified_end_frames = pd.to_numeric(
        segment_manifest[
            "verified_end_frame"
        ],
        errors="coerce",
    )

    invalid_count_mask = (
        verified_frame_counts.isna()
        | verified_frame_counts.lt(1)
        | verified_frame_counts.mod(1).ne(0)
    )

    invalid_range_mask = (
        verified_start_frames.isna()
        | verified_end_frames.isna()
        | verified_start_frames.lt(0)
        | verified_end_frames.lt(
            verified_start_frames
        )
    )

    if invalid_count_mask.any():
        raise ValueError(
            "Invalid verified frame counts found: "
            f"{int(invalid_count_mask.sum())}"
        )

    if invalid_range_mask.any():
        raise ValueError(
            "Invalid verified temporal ranges found: "
            f"{int(invalid_range_mask.sum())}"
        )

    has_temporal_extent = (
        verified_end_frames
        .gt(
            verified_start_frames
        )
    )

    temporal_grounding_candidate = (
        verified_frame_counts
        .ge(
            minimum_frames
        )
        & has_temporal_extent
    )

    temporal_supervision_level = np.select(
        [
            temporal_grounding_candidate,
            verified_frame_counts.eq(1),
        ],
        [
            "multi_frame_weak_supervision",
            "single_frame_anchor_only",
        ],
        default=(
            "insufficient_multi_frame_supervision"
        ),
    )

    return (
        segment_manifest
        .assign(
            verified_frame_count=(
                verified_frame_counts
                .astype("Int64")
            ),

            temporal_grounding_candidate=(
                temporal_grounding_candidate
                .astype("boolean")
            ),

            temporal_supervision_level=(
                pd.Series(
                    temporal_supervision_level,
                    index=segment_manifest.index,
                    dtype="string",
                )
            ),

            # Kvasir-Capsule provides verified positive
            # frames, not exact clinical onset/offset.
            verified_temporal_boundary_available=False,
        )
    )


finding_segment_manifest = (
    add_temporal_supervision_metadata(
        segment_manifest=(
            finding_segment_manifest
        ),
        config=CONFIG,
    )
)

### 25. Build temporal evidence windows

In [ ]:
TEMPORAL_MANIFEST_COLUMNS = [
    "finding_segment_id",
    "video_key",
    "video_relpath",
    "finding_class",
    "finding_class_normalized",
    "split",
    "target_frame_number",
    "target_opencv_frame_index",
    "temporal_offset_seconds",
    "effective_temporal_offset_seconds",
    "frame_delta",
    "context_frame_number",
    "opencv_frame_index",
    "is_target",
    "requested_context_count",
    "available_context_count",
    "temporal_window_complete",
]


def build_temporal_manifest(
    segment_manifest,
    video_manifest,
    config,
    frame_index_offset,
):
    """
    Builds temporal evidence references around eligible
    verified finding segments.

    One row represents one valid temporal context frame.
    No images are extracted.
    """

    required_segment_columns = {
        "finding_segment_id",
        "video_key",
        "finding_class",
        "finding_class_normalized",
        "split",
        "target_frame_number",
        "target_opencv_frame_index",
        "temporal_grounding_candidate",
    }

    required_video_columns = {
        "video_key",
        "video_relpath",
        "frame_count",
        "container_fps",
    }

    missing_segment_columns = sorted(
        required_segment_columns
        - set(segment_manifest.columns)
    )

    missing_video_columns = sorted(
        required_video_columns
        - set(video_manifest.columns)
    )

    if missing_segment_columns:
        raise KeyError(
            "Temporal manifest is missing segment columns: "
            f"{missing_segment_columns}"
        )

    if missing_video_columns:
        raise KeyError(
            "Temporal manifest is missing video columns: "
            f"{missing_video_columns}"
        )

    if not video_manifest[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Video manifest contains duplicate video keys."
        )

    candidate_flags = (
        segment_manifest[
            "temporal_grounding_candidate"
        ]
        .astype("boolean")
    )

    if candidate_flags.isna().any():
        raise ValueError(
            "Temporal-grounding candidate values "
            "contain missing entries."
        )

    eligible_segments = (
        segment_manifest.loc[
            candidate_flags
        ]
        .copy()
    )

    if eligible_segments.empty:
        return pd.DataFrame(
            columns=TEMPORAL_MANIFEST_COLUMNS
        )

    raw_offsets = pd.Series(
        config[
            "temporal_context_offsets_seconds"
        ],
        dtype="object",
    )

    temporal_offsets = pd.to_numeric(
        raw_offsets,
        errors="coerce",
    )

    if (
        temporal_offsets.empty
        or temporal_offsets.isna().any()
        or not np.isfinite(
            temporal_offsets
        ).all()
    ):
        raise ValueError(
            "Temporal context offsets must be "
            "finite numeric values."
        )

    temporal_offsets = (
        temporal_offsets
        .astype("float64")
    )

    temporal_offsets = temporal_offsets.mask(
        np.isclose(
            temporal_offsets,
            0.0,
        ),
        0.0,
    )

    if temporal_offsets.duplicated().any():
        raise ValueError(
            "Temporal context offsets contain "
            "duplicate values."
        )

    if not temporal_offsets.eq(0.0).any():
        raise ValueError(
            "Temporal context offsets must include 0 "
            "for the target frame."
        )

    offset_table = (
        pd.DataFrame(
            {
                "temporal_offset_seconds":
                    temporal_offsets
            }
        )
        .sort_values(
            "temporal_offset_seconds"
        )
        .reset_index(
            drop=True
        )
    )

    video_metadata = (
        video_manifest[
            [
                "video_key",
                "video_relpath",
                "frame_count",
                "container_fps",
            ]
        ]
        .copy()
        .assign(
            frame_count=lambda data:
                pd.to_numeric(
                    data["frame_count"],
                    errors="coerce",
                ),

            container_fps=lambda data:
                pd.to_numeric(
                    data["container_fps"],
                    errors="coerce",
                ),
        )
    )

    invalid_video_metadata = (
        video_metadata["frame_count"].isna()
        | video_metadata["frame_count"].le(0)
        | video_metadata["frame_count"].mod(1).ne(0)
        | video_metadata["container_fps"].isna()
        | video_metadata["container_fps"].le(0)
        | ~np.isfinite(
            video_metadata["container_fps"]
        )
    )

    if invalid_video_metadata.any():

        invalid_video_keys = (
            video_metadata.loc[
                invalid_video_metadata,
                "video_key",
            ]
            .tolist()
        )

        raise ValueError(
            "Invalid video frame count or FPS for: "
            f"{invalid_video_keys}"
        )

    video_metadata = (
        video_metadata
        .assign(
            frame_count=lambda data:
                data["frame_count"]
                .astype("Int64")
        )
    )

    segment_video_metadata = (
        eligible_segments
        .merge(
            video_metadata,
            on="video_key",
            how="left",
            validate="many_to_one",
            indicator=True,
        )
    )

    missing_video_mask = (
        segment_video_metadata[
            "_merge"
        ]
        .ne("both")
    )

    if missing_video_mask.any():

        missing_video_keys = (
            segment_video_metadata.loc[
                missing_video_mask,
                "video_key",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise KeyError(
            "Segment videos are missing from the "
            f"video manifest: {missing_video_keys}"
        )

    segment_video_metadata = (
        segment_video_metadata
        .drop(
            columns="_merge"
        )
    )

    expanded = (
        segment_video_metadata
        .merge(
            offset_table,
            how="cross",
        )
    )

    expanded = (
        expanded
        .assign(
            frame_delta=lambda data:
                (
                    data[
                        "temporal_offset_seconds"
                    ]
                    * data["container_fps"]
                )
                .round()
                .astype("Int64")
        )
    )

    duplicate_frame_delta_mask = (
        expanded.duplicated(
            subset=[
                "finding_segment_id",
                "frame_delta",
            ],
            keep=False,
        )
    )

    if duplicate_frame_delta_mask.any():

        duplicate_offsets = (
            expanded.loc[
                duplicate_frame_delta_mask,
                [
                    "finding_segment_id",
                    "temporal_offset_seconds",
                    "frame_delta",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Different temporal offsets resolve to "
            "the same frame: "
            f"{duplicate_offsets}"
        )

    frame_index_offset = int(
        frame_index_offset
    )

    expanded = (
        expanded
        .assign(
            opencv_frame_index=lambda data:
                (
                    data[
                        "target_opencv_frame_index"
                    ]
                    + data["frame_delta"]
                )
                .astype("Int64"),

            effective_temporal_offset_seconds=lambda data:
                (
                    data["frame_delta"]
                    / data["container_fps"]
                ),
        )
        .assign(
            context_frame_number=lambda data:
                (
                    data["opencv_frame_index"]
                    - frame_index_offset
                )
                .astype("Int64"),

            is_target=lambda data:
                data[
                    "temporal_offset_seconds"
                ]
                .eq(0.0),
        )
    )

    expanded = (
        expanded
        .assign(
            context_available=lambda data:
                (
                    data["opencv_frame_index"].ge(0)
                    & data["opencv_frame_index"].lt(
                        data["frame_count"]
                    )
                    & data[
                        "context_frame_number"
                    ].ge(0)
                )
        )
    )

    window_statistics = (
        expanded
        .groupby(
            "finding_segment_id",
            sort=False,
            observed=True,
        )
        .agg(
            requested_context_count=(
                "context_available",
                "size",
            ),
            available_context_count=(
                "context_available",
                "sum",
            ),
            temporal_window_complete=(
                "context_available",
                "all",
            ),
        )
        .reset_index()
    )

    temporal_manifest = (
        expanded.loc[
            expanded[
                "context_available"
            ]
        ]
        .merge(
            window_statistics,
            on="finding_segment_id",
            how="left",
            validate="many_to_one",
        )
        .loc[
            :,
            TEMPORAL_MANIFEST_COLUMNS,
        ]
        .sort_values(
            [
                "finding_segment_id",
                "temporal_offset_seconds",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return temporal_manifest


if CONFIG[
    "temporal_context_enabled"
]:
    temporal_manifest = (
        build_temporal_manifest(
            segment_manifest=(
                finding_segment_manifest
            ),
            video_manifest=video_manifest,
            config=CONFIG,
            frame_index_offset=(
                FRAME_INDEX_OFFSET
            ),
        )
    )

else:
    temporal_manifest = pd.DataFrame(
        columns=(
            TEMPORAL_MANIFEST_COLUMNS
        )
    )

temporal_window_report = (
    temporal_manifest[
        [
            "finding_segment_id",
            "temporal_window_complete",
        ]
    ]
    .drop_duplicates()
)


print(
    "Temporal evidence frame references:",
    f"{len(temporal_manifest):,}",
)

print(
    "Temporal evidence windows:",
    f"{len(temporal_window_report):,}",
)

print(
    "Incomplete temporal windows:",
    int(
        (
            ~temporal_window_report[
                "temporal_window_complete"
            ]
        )
        .sum()
    ),
)


display(
    temporal_manifest.head()
)

### 26. Determine whether each temporal frame is verified or unknown

In [ ]:
def add_temporal_context_metadata(
    temporal_manifest,
    labelled_manifest,
):
    """
    Adds annotation status and image-source metadata
    to temporal context-frame references.

    An unlabelled context means that no ground-truth
    annotation is available. It does not mean normal.
    """

    required_temporal_columns = {
        "video_key",
        "context_frame_number",
        "finding_class_normalized",
        "is_target",
    }

    required_labelled_columns = {
        "video_key",
        "frame_number",
        "finding_class_normalized",
        "image_path",
        "image_relpath",
    }

    missing_temporal_columns = sorted(
        required_temporal_columns
        - set(temporal_manifest.columns)
    )

    missing_labelled_columns = sorted(
        required_labelled_columns
        - set(labelled_manifest.columns)
    )

    if missing_temporal_columns:
        raise KeyError(
            "Temporal context metadata is missing columns: "
            f"{missing_temporal_columns}"
        )

    if missing_labelled_columns:
        raise KeyError(
            "Labelled-frame metadata is missing columns: "
            f"{missing_labelled_columns}"
        )

    verified_frames = (
        labelled_manifest[
            [
                "video_key",
                "frame_number",
            ]
        ]
        .drop_duplicates()
        .rename(
            columns={
                "frame_number":
                    "context_frame_number"
            }
        )
        .assign(
            context_is_verified=True
        )
    )

    verified_findings = (
        labelled_manifest[
            [
                "video_key",
                "frame_number",
                "finding_class_normalized",
            ]
        ]
        .drop_duplicates()
        .rename(
            columns={
                "frame_number":
                    "context_frame_number"
            }
        )
        .assign(
            context_is_verified_same_finding=True
        )
    )

    image_path_consistency = (
        labelled_manifest
        .groupby(
            [
                "video_key",
                "frame_number",
            ],
            sort=False,
            observed=True,
        )[
            [
                "image_path",
                "image_relpath",
            ]
        ]
        .nunique(
            dropna=False
        )
    )

    inconsistent_image_mask = (
        image_path_consistency
        .gt(1)
        .any(axis=1)
    )

    if inconsistent_image_mask.any():

        inconsistent_frames = (
            image_path_consistency.index[
                inconsistent_image_mask
            ]
            .tolist()
        )

        raise ValueError(
            "Conflicting official image paths found for "
            f"verified frames: {inconsistent_frames}"
        )

    official_frame_images = (
        labelled_manifest
        .sort_values(
            [
                "video_key",
                "frame_number",
                "finding_class_normalized",
            ],
            kind="stable",
        )
        .groupby(
            [
                "video_key",
                "frame_number",
            ],
            as_index=False,
            sort=False,
            observed=True,
        )
        .agg(
            context_official_image_path=(
                "image_path",
                "first",
            ),

            context_official_image_relpath=(
                "image_relpath",
                "first",
            ),

            context_verified_labels=(
                "finding_class_normalized",
                lambda values:
                    "|".join(
                        sorted(
                            values
                            .dropna()
                            .astype("string")
                            .unique()
                            .tolist()
                        )
                    ),
            ),
        )
        .rename(
            columns={
                "frame_number":
                    "context_frame_number"
            }
        )
    )

    result = (
        temporal_manifest
        .merge(
            verified_frames,
            on=[
                "video_key",
                "context_frame_number",
            ],
            how="left",
            validate="many_to_one",
        )
        .merge(
            verified_findings,
            on=[
                "video_key",
                "context_frame_number",
                "finding_class_normalized",
            ],
            how="left",
            validate="many_to_one",
        )
        .merge(
            official_frame_images,
            on=[
                "video_key",
                "context_frame_number",
            ],
            how="left",
            validate="many_to_one",
        )
        .assign(
            context_is_verified=lambda data:
                data[
                    "context_is_verified"
                ]
                .fillna(False)
                .astype("boolean"),

            context_is_verified_same_finding=lambda data:
                data[
                    "context_is_verified_same_finding"
                ]
                .fillna(False)
                .astype("boolean"),
        )
    )

    invalid_target_mask = (
        result["is_target"]
        .astype("boolean")
        &
        (
            ~result[
                "context_is_verified_same_finding"
            ]
            | result[
                "context_official_image_path"
            ].isna()
        )
    )

    if invalid_target_mask.any():

        invalid_targets = (
            result.loc[
                invalid_target_mask,
                [
                    "finding_segment_id",
                    "video_key",
                    "context_frame_number",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise RuntimeError(
            "Temporal targets could not be matched to "
            "their official verified images: "
            f"{invalid_targets}"
        )

    annotation_status = np.select(
        [
            result[
                "is_target"
            ].astype(bool),

            result[
                "context_is_verified_same_finding"
            ].astype(bool),

            result[
                "context_is_verified"
            ].astype(bool),
        ],
        [
            "verified_target",
            "verified_same_finding",
            "verified_other_finding",
        ],
        default="unlabelled_context",
    )

    return (
        result
        .assign(
            context_annotation_status=(
                pd.Series(
                    annotation_status,
                    index=result.index,
                    dtype="string",
                )
            ),

            context_image_path=(
                result[
                    "context_official_image_path"
                ]
            ),

            context_image_relpath=(
                result[
                    "context_official_image_relpath"
                ]
            ),

            context_image_source=(
                pd.Series(
                    np.where(
                        result[
                            "context_official_image_path"
                        ].notna(),
                        "official_labelled_image",
                        "video_extraction_required",
                    ),
                    index=result.index,
                    dtype="string",
                )
            ),
        )
    )

temporal_manifest = (
    add_temporal_context_metadata(
        temporal_manifest=(
            temporal_manifest
        ),
        labelled_manifest=(
            labelled_frame_manifest
        ),
    )
)


display(
    temporal_manifest[
        "context_image_source"
    ].value_counts(
        dropna=False
    )
)

### 27. Prepare only missing temporal frames for extraction

In [ ]:
TEMPORAL_EXTRACTION_REQUEST_COLUMNS = [
    "video_key",
    "opencv_frame_index",
    "video_path",
    "output_path",
]


def build_temporal_output_path(
    video_key,
    opencv_frame_index,
    dirs,
    config,
):
    """
    Builds the deterministic output path for one
    temporal context frame.
    """

    extension = (
        str(
            config[
                "temporal_context_image_format"
            ]
        )
        .strip()
        .lower()
        .lstrip(".")
    )

    if extension not in {
        "jpg",
        "jpeg",
        "png",
    }:
        raise ValueError(
            "Unsupported temporal context image format: "
            f"{extension}"
        )

    return (
        Path(
            dirs[
                "temporal_frames_dir"
            ]
        )
        / str(video_key)
        / (
            f"frame_"
            f"{int(opencv_frame_index):08d}"
            f".{extension}"
        )
    )


def build_temporal_extraction_requests(
    temporal_manifest,
    video_manifest,
    dirs,
    config,
):
    """
    Builds one extraction request per unique temporal
    context frame requiring an image from the source video.

    This function plans extraction but performs no disk I/O.
    """

    required_temporal_columns = {
        "video_key",
        "opencv_frame_index",
        "context_image_path",
        "context_image_source",
    }

    required_video_columns = {
        "video_key",
        "video_path",
    }

    missing_temporal_columns = sorted(
        required_temporal_columns
        - set(temporal_manifest.columns)
    )

    missing_video_columns = sorted(
        required_video_columns
        - set(video_manifest.columns)
    )

    if missing_temporal_columns:
        raise KeyError(
            "Temporal extraction planning is missing "
            f"columns: {missing_temporal_columns}"
        )

    if missing_video_columns:
        raise KeyError(
            "Video manifest is missing columns: "
            f"{missing_video_columns}"
        )

    if not video_manifest[
        "video_key"
    ].is_unique:
        raise ValueError(
            "Video manifest contains duplicate video keys."
        )

    extraction_mask = (
        temporal_manifest[
            "context_image_source"
        ]
        .astype("string")
        .eq(
            "video_extraction_required"
        )
        .fillna(False)
    )

    inconsistent_mask = (
        extraction_mask
        & temporal_manifest[
            "context_image_path"
        ].notna()
    )

    if inconsistent_mask.any():
        raise ValueError(
            "Some frames are marked as requiring video "
            "extraction but already have an image path."
        )

    requests = (
        temporal_manifest.loc[
            extraction_mask,
            [
                "video_key",
                "opencv_frame_index",
            ],
        ]
        .drop_duplicates()
        .reset_index(
            drop=True
        )
    )

    if requests.empty:
        return pd.DataFrame(
            columns=(
                TEMPORAL_EXTRACTION_REQUEST_COLUMNS
            )
        )

    frame_indices = pd.to_numeric(
        requests[
            "opencv_frame_index"
        ],
        errors="coerce",
    )

    invalid_index_mask = (
        frame_indices.isna()
        | ~np.isfinite(
            frame_indices.astype("float64")
        )
        | frame_indices.lt(0)
        | frame_indices.mod(1).ne(0)
    )

    if invalid_index_mask.any():
        raise ValueError(
            "Temporal extraction requests contain "
            "invalid OpenCV frame indices."
        )

    requests = (
        requests
        .assign(
            opencv_frame_index=(
                frame_indices.astype("Int64")
            )
        )
        .merge(
            video_manifest[
                [
                    "video_key",
                    "video_path",
                ]
            ],
            on="video_key",
            how="left",
            validate="many_to_one",
            indicator=True,
        )
    )

    missing_video_mask = (
        requests[
            "_merge"
        ]
        .ne("both")
        | requests[
            "video_path"
        ].isna()
    )

    if missing_video_mask.any():
        missing_video_keys = (
            requests.loc[
                missing_video_mask,
                "video_key",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise KeyError(
            "Videos required for temporal extraction "
            "are missing: "
            f"{missing_video_keys}"
        )

    requests = (
        requests
        .drop(
            columns="_merge"
        )
        .assign(
            output_path=lambda data: [
                str(
                    build_temporal_output_path(
                        video_key=video_key,
                        opencv_frame_index=(
                            opencv_frame_index
                        ),
                        dirs=dirs,
                        config=config,
                    )
                )
                for video_key, opencv_frame_index
                in zip(
                    data["video_key"],
                    data["opencv_frame_index"],
                )
            ]
        )
        .loc[
            :,
            TEMPORAL_EXTRACTION_REQUEST_COLUMNS,
        ]
        .sort_values(
            [
                "video_key",
                "opencv_frame_index",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    if requests[
        "output_path"
    ].duplicated().any():
        raise RuntimeError(
            "Multiple temporal frames resolve to the "
            "same output path."
        )

    return requests


temporal_extraction_requests = (
    build_temporal_extraction_requests(
        temporal_manifest=temporal_manifest,
        video_manifest=video_manifest,
        dirs=DIRS,
        config=CONFIG,
    )
)


print(
    "Unique temporal frames requiring extraction:",
    f"{len(temporal_extraction_requests):,}",
)

display(
    temporal_extraction_requests.head()
)

### 28. Run temporal frame extraction

In [ ]:
TEMPORAL_EXTRACTION_RESULT_COLUMNS = [
    "video_key",
    "opencv_frame_index",
    "extracted_image_path",
    "extraction_status",
    "extraction_error",
]


def get_temporal_image_write_parameters(
    output_path,
    config,
):
    """
    Returns the OpenCV encoding parameters appropriate
    for the requested output format.
    """

    extension = output_path.suffix.lower()

    if extension in {
        ".jpg",
        ".jpeg",
    }:
        quality = int(
            config[
                "temporal_context_jpeg_quality"
            ]
        )

        if not 0 <= quality <= 100:
            raise ValueError(
                "JPEG quality must be between 0 and 100."
            )

        return [
            cv2.IMWRITE_JPEG_QUALITY,
            quality,
        ]

    if extension == ".png":
        compression = int(
            config.get(
                "temporal_context_png_compression",
                3,
            )
        )

        if not 0 <= compression <= 9:
            raise ValueError(
                "PNG compression must be between 0 and 9."
            )

        return [
            cv2.IMWRITE_PNG_COMPRESSION,
            compression,
        ]

    raise ValueError(
        f"Unsupported temporal image format: {extension}"
    )


def extract_temporal_frames_for_video(
    requests,
    config,
):
    """
    Executes temporal frame extraction for exactly one
    source video.

    Frames are processed sequentially and written directly
    to their declared output paths. Image arrays are not
    retained after each request is processed.

    Returns:
        One extraction-result record per requested frame.
    """

    if requests.empty:
        return pd.DataFrame(
            columns=(
                TEMPORAL_EXTRACTION_RESULT_COLUMNS
            )
        )

    required_columns = {
        "video_key",
        "video_path",
        "opencv_frame_index",
        "output_path",
    }

    missing_columns = sorted(
        required_columns
        - set(requests.columns)
    )

    if missing_columns:
        raise KeyError(
            "Temporal extraction batch is missing "
            f"columns: {missing_columns}"
        )

    video_keys = (
        requests[
            "video_key"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_video_key_mask = (
        video_keys.isna()
        | video_keys.eq("")
    )

    if invalid_video_key_mask.any():
        raise ValueError(
            "Temporal extraction batch contains "
            "missing or empty video keys."
        )

    if video_keys.nunique() != 1:
        raise ValueError(
            "One temporal extraction batch must contain "
            "exactly one video key."
        )

    video_paths = (
        requests[
            "video_path"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_video_path_mask = (
        video_paths.isna()
        | video_paths.eq("")
    )

    if invalid_video_path_mask.any():
        raise ValueError(
            "Temporal extraction batch contains "
            "missing video paths."
        )

    if video_paths.nunique() != 1:
        raise ValueError(
            "One temporal extraction batch must reference "
            "exactly one physical video path."
        )

    video_path = Path(
        video_paths.iloc[0]
    )

    if not video_path.is_file():
        raise FileNotFoundError(
            "Temporal extraction source video does not "
            f"exist: {video_path}"
        )

    frame_indices = pd.to_numeric(
        requests[
            "opencv_frame_index"
        ],
        errors="coerce",
    )

    invalid_frame_index_mask = (
        frame_indices.isna()
        | ~np.isfinite(
            frame_indices.astype(
                "float64"
            )
        )
        | frame_indices.lt(0)
        | frame_indices.mod(1).ne(0)
    )

    if invalid_frame_index_mask.any():
        invalid_indices = (
            requests.loc[
                invalid_frame_index_mask,
                "opencv_frame_index",
            ]
            .tolist()
        )

        raise ValueError(
            "Temporal extraction batch contains invalid "
            f"OpenCV frame indices: {invalid_indices}"
        )

    output_paths = (
        requests[
            "output_path"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_output_path_mask = (
        output_paths.isna()
        | output_paths.eq("")
    )

    if invalid_output_path_mask.any():
        raise ValueError(
            "Temporal extraction batch contains "
            "missing output paths."
        )

    normalized_requests = (
        requests
        .assign(
            video_key=video_keys,

            video_path=video_paths,

            opencv_frame_index=(
                frame_indices.astype(
                    "Int64"
                )
            ),

            output_path=output_paths,
        )
        .sort_values(
            "opencv_frame_index",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    duplicate_frame_mask = (
        normalized_requests.duplicated(
            subset=[
                "video_key",
                "opencv_frame_index",
            ],
            keep=False,
        )
    )

    if duplicate_frame_mask.any():
        duplicate_frames = (
            normalized_requests.loc[
                duplicate_frame_mask,
                [
                    "video_key",
                    "opencv_frame_index",
                ],
            ]
            .drop_duplicates()
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Duplicate temporal extraction requests "
            f"found: {duplicate_frames}"
        )

    duplicate_output_path_mask = (
        normalized_requests[
            "output_path"
        ]
        .duplicated(
            keep=False
        )
    )

    if duplicate_output_path_mask.any():
        duplicate_paths = (
            normalized_requests.loc[
                duplicate_output_path_mask,
                "output_path",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Multiple requested frames resolve to the "
            f"same output path: {duplicate_paths}"
        )

    capture = cv2.VideoCapture(
        str(video_path)
    )

    if not capture.isOpened():
        capture.release()

        raise RuntimeError(
            "OpenCV could not open the temporal "
            f"source video: {video_path}"
        )

    records = []

    try:
        for request in (
            normalized_requests.itertuples(
                index=False
            )
        ):
            frame_index = int(
                request.opencv_frame_index
            )

            output_path = Path(
                request.output_path
            )

            success = False
            status = None
            error_message = None

            try:
                output_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            except OSError as error:
                status = (
                    "directory_creation_failed"
                )

                error_message = str(
                    error
                )

            if status is None:
                existing_output_available = (
                    output_path.is_file()
                    and output_path.stat().st_size > 0
                )

                if existing_output_available:
                    success = True
                    status = "reused_existing"

                else:
                    seek_success = capture.set(
                        cv2.CAP_PROP_POS_FRAMES,
                        frame_index,
                    )

                    if not seek_success:
                        status = "seek_failed"

                        error_message = (
                            "OpenCV could not seek to "
                            f"frame {frame_index}."
                        )

                    else:
                        readable, frame = (
                            capture.read()
                        )

                        if (
                            not readable
                            or frame is None
                            or frame.size == 0
                        ):
                            status = "read_failed"

                            error_message = (
                                "OpenCV could not decode "
                                f"frame {frame_index}."
                            )

                        else:
                            write_parameters = (
                                get_temporal_image_write_parameters(
                                    output_path=output_path,
                                    config=config,
                                )
                            )

                            try:
                                write_success = (
                                    cv2.imwrite(
                                        str(
                                            output_path
                                        ),
                                        frame,
                                        write_parameters,
                                    )
                                )

                            except (
                                cv2.error,
                                OSError,
                            ) as error:
                                write_success = False
                                error_message = str(
                                    error
                                )

                            success = bool(
                                write_success
                                and output_path.is_file()
                                and output_path.stat().st_size
                                > 0
                            )

                            if success:
                                status = "extracted"
                                error_message = None

                            else:
                                status = "write_failed"

                                if error_message is None:
                                    error_message = (
                                        "OpenCV did not "
                                        "write a valid "
                                        "image file."
                                    )

            records.append(
                {
                    "video_key":
                        request.video_key,

                    "opencv_frame_index":
                        frame_index,

                    "extracted_image_path":
                        (
                            str(output_path)
                            if success
                            else None
                        ),

                    "extraction_status":
                        status,

                    "extraction_error":
                        error_message,
                }
            )

    finally:
        capture.release()

    return pd.DataFrame.from_records(
        records,
        columns=(
            TEMPORAL_EXTRACTION_RESULT_COLUMNS
        ),
    )

### 29. Temporal frame extraction

In [ ]:
def run_temporal_extraction(
    extraction_requests,
    config,
):
    """
    Executes temporal frame extraction grouped by video.

    Extracted images are written to the output paths
    already present in extraction_requests.

    Returns:
        One result row per unique requested frame.
    """

    empty_result = pd.DataFrame(
        columns=(
            TEMPORAL_EXTRACTION_RESULT_COLUMNS
        )
    )

    required_config_keys = {
        "temporal_context_enabled",
        "temporal_context_extract_frames",
    }

    missing_config_keys = sorted(
        required_config_keys
        - set(config)
    )

    if missing_config_keys:
        raise KeyError(
            "Temporal extraction configuration is missing "
            f"keys: {missing_config_keys}"
        )

    if (
        not config[
            "temporal_context_enabled"
        ]
        or not config[
            "temporal_context_extract_frames"
        ]
    ):
        return empty_result

    if extraction_requests.empty:
        return empty_result

    required_columns = {
        "video_key",
        "video_path",
        "opencv_frame_index",
        "output_path",
    }

    missing_columns = sorted(
        required_columns
        - set(extraction_requests.columns)
    )

    if missing_columns:
        raise KeyError(
            "Temporal extraction requests are missing "
            f"columns: {missing_columns}"
        )

    extracted_tables = [
        extract_temporal_frames_for_video(
            requests=video_requests,
            config=config,
        )

        for _, video_requests
        in tqdm(
            extraction_requests.groupby(
                "video_key",
                sort=False,
                observed=True,
            ),
            desc="Extracting temporal evidence",
        )
    ]

    extracted_tables = [
        table
        for table in extracted_tables
        if not table.empty
    ]

    if not extracted_tables:
        return empty_result

    extracted_frames = (
        pd.concat(
            extracted_tables,
            ignore_index=True,
        )
        .sort_values(
            [
                "video_key",
                "opencv_frame_index",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    duplicate_result_mask = (
        extracted_frames.duplicated(
            subset=[
                "video_key",
                "opencv_frame_index",
            ],
            keep=False,
        )
    )

    if duplicate_result_mask.any():
        duplicate_results = (
            extracted_frames.loc[
                duplicate_result_mask,
                [
                    "video_key",
                    "opencv_frame_index",
                ],
            ]
            .drop_duplicates()
            .to_dict(
                orient="records"
            )
        )

        raise RuntimeError(
            "Temporal extraction produced duplicate "
            f"frame results: {duplicate_results}"
        )

    return extracted_frames


extracted_temporal_frames = (
    run_temporal_extraction(
        extraction_requests=(
            temporal_extraction_requests
        ),
        config=CONFIG,
    )
)


print(
    "Temporal extraction result rows:",
    f"{len(extracted_temporal_frames):,}",
)


display(
    extracted_temporal_frames[
        "extraction_status"
    ].value_counts(
        dropna=False
    )
)

### 30. Adding temporal extracted images to the manifest

In [ ]:
def attach_temporal_extraction_results(
    temporal_manifest,
    extraction_results,
):
    """
    Attaches temporal extraction results to the manifest.

    Official labelled images are preferred over images
    extracted from video.
    """

    required_temporal_columns = {
        "video_key",
        "opencv_frame_index",
        "context_official_image_path",
        "context_image_path",
    }

    required_result_columns = {
        "video_key",
        "opencv_frame_index",
        "extracted_image_path",
        "extraction_status",
        "extraction_error",
    }

    missing_temporal_columns = sorted(
        required_temporal_columns
        - set(temporal_manifest.columns)
    )

    missing_result_columns = sorted(
        required_result_columns
        - set(extraction_results.columns)
    )

    if missing_temporal_columns:
        raise KeyError(
            "Temporal manifest is missing columns: "
            f"{missing_temporal_columns}"
        )

    if missing_result_columns:
        raise KeyError(
            "Temporal extraction results are missing "
            f"columns: {missing_result_columns}"
        )

    duplicate_result_mask = (
        extraction_results.duplicated(
            subset=[
                "video_key",
                "opencv_frame_index",
            ],
            keep=False,
        )
    )

    if duplicate_result_mask.any():
        raise ValueError(
            "Temporal extraction results contain "
            "duplicate video-frame keys."
        )

    # Remove results from a previous execution so that
    # rerunning the cell does not create _x/_y columns.
    existing_extraction_columns = [
        column
        for column in [
            "extracted_image_path",
            "extraction_status",
            "extraction_error",
        ]
        if column in temporal_manifest.columns
    ]

    result = (
        temporal_manifest
        .drop(
            columns=existing_extraction_columns
        )
        .merge(
            extraction_results[
                [
                    "video_key",
                    "opencv_frame_index",
                    "extracted_image_path",
                    "extraction_status",
                    "extraction_error",
                ]
            ],
            on=[
                "video_key",
                "opencv_frame_index",
            ],
            how="left",
            validate="many_to_one",
        )
    )

    final_context_paths = (
        result[
            "context_official_image_path"
        ]
        .combine_first(
            result[
                "extracted_image_path"
            ]
        )
        .combine_first(
            result[
                "context_image_path"
            ]
        )
    )

    context_sources = np.select(
        [
            result[
                "context_official_image_path"
            ].notna(),

            final_context_paths.notna(),
        ],
        [
            "official_labelled_image",
            "extracted_from_video",
        ],
        default="missing",
    )

    return (
        result
        .assign(
            context_image_path=(
                final_context_paths
            ),

            context_image_source=(
                pd.Series(
                    context_sources,
                    index=result.index,
                    dtype="string",
                )
            ),
        )
    )


if (
    CONFIG[
        "temporal_context_enabled"
    ]
    and CONFIG[
        "temporal_context_extract_frames"
    ]
):
    temporal_manifest = (
        attach_temporal_extraction_results(
            temporal_manifest=(
                temporal_manifest
            ),
            extraction_results=(
                extracted_temporal_frames
            ),
        )
    )


def path_to_storage_relpath_or_missing(
    path,
    storage_root,
):
    """
    Converts an absolute storage path into a path relative
    to the declared storage root while preserving missing
    values.
    """

    if pd.isna(path):
        return pd.NA

    return path_relative_to_storage(
        value=path,
        storage_root=storage_root,
    )


temporal_manifest = (
    temporal_manifest
    .assign(
        context_image_relpath=lambda data:
            data[
                "context_image_path"
            ]
            .map(
                lambda path:
                    path_to_storage_relpath_or_missing(
                        path=path,
                        storage_root=CONFIG[
                            "storage_root"
                        ],
                    )
            )
            .astype("string")
    )
)

### 31. Validate final temporal artifacts


In [ ]:
# ------------------------------------------------------------------
# Validate temporal extraction
# ------------------------------------------------------------------

TEMPORAL_EXTRACTION_FAILURE_COLUMNS = [
    "finding_segment_id",
    "video_key",
    "opencv_frame_index",
    "temporal_offset_seconds",
    "context_image_path",
    "context_image_source",
    "extraction_status",
    "extraction_error",
]


temporal_extraction_failure_path = (
    Path(
        DIRS[
            "reports_dir"
        ]
    )
    / "temporal_extraction_failures.parquet"
)


if CONFIG[
    "temporal_context_enabled"
]:

    required_temporal_validation_columns = {
        "finding_segment_id",
        "video_key",
        "opencv_frame_index",
        "temporal_offset_seconds",
        "context_image_path",
    }

    missing_validation_columns = sorted(
        required_temporal_validation_columns
        - set(temporal_manifest.columns)
    )

    if missing_validation_columns:
        raise KeyError(
            "Temporal manifest cannot be validated. "
            f"Missing columns: {missing_validation_columns}"
        )


    missing_context_mask = (
        temporal_manifest[
            "context_image_path"
        ]
        .isna()
    )


    missing_context_count = int(
        missing_context_mask.sum()
    )


    temporal_extraction_failures = (
        temporal_manifest.loc[
            missing_context_mask
        ]
        .reindex(
            columns=(
                TEMPORAL_EXTRACTION_FAILURE_COLUMNS
            )
        )
        .reset_index(
            drop=True
        )
    )


    print(
        "Temporal context rows:",
        f"{len(temporal_manifest):,}",
    )

    print(
        "Temporal context images missing:",
        f"{missing_context_count:,}",
    )


    # The failure report is persisted here because the
    # pipeline may stop before reaching the final
    # Data Lake persistence cell.
    if CONFIG[
        "temporal_context_extract_frames"
    ]:

        temporal_extraction_failure_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        temporal_extraction_failures.to_parquet(
            temporal_extraction_failure_path,
            index=False,
            engine="pyarrow",
            compression="zstd",
        )


        if missing_context_count > 0:
            raise RuntimeError(
                "Temporal extraction finished with "
                f"{missing_context_count:,} missing images. "
                "See the temporal extraction failure report: "
                f"{temporal_extraction_failure_path}"
            )

else:

    missing_context_count = 0

    temporal_extraction_failures = pd.DataFrame(
        columns=(
            TEMPORAL_EXTRACTION_FAILURE_COLUMNS
        )
    )

    print(
        "Temporal context is disabled."
    )

### 32. Image quality metrics

In [ ]:
IMAGE_QC_COLUMNS = [
    "image_path",
    "image_readable",
    "image_read_error",
    "image_width",
    "image_height",
    "brightness_mean",
    "contrast_std",
    "blur_laplacian",
    "underexposed_fraction",
    "overexposed_fraction",
    "specular_fraction",
]


def build_capsule_fov_mask(
    image_height,
    image_width,
    radius_fraction,
):
    """
    Builds a circular field-of-view mask that excludes
    the black border around a capsule-endoscopy frame.
    """

    if not 0 < radius_fraction <= 1:
        raise ValueError(
            "qc_fov_radius_fraction must be in (0, 1]."
        )

    center_y = (
        image_height - 1
    ) / 2.0

    center_x = (
        image_width - 1
    ) / 2.0

    radius = (
        min(
            image_height,
            image_width,
        )
        / 2.0
        * radius_fraction
    )

    y_coordinates, x_coordinates = np.ogrid[
        :image_height,
        :image_width,
    ]

    return (
        (
            x_coordinates
            - center_x
        )
        ** 2
        +
        (
            y_coordinates
            - center_y
        )
        ** 2
        <= radius ** 2
    )


def build_unreadable_image_qc_record(
    image_path,
    error_message,
):
    """
    Builds a stable QC record for an unreadable image.
    """

    return {
        "image_path":
            (
                str(image_path)
                if not pd.isna(image_path)
                else pd.NA
            ),

        "image_readable":
            False,

        "image_read_error":
            error_message,

        "image_width":
            np.nan,

        "image_height":
            np.nan,

        "brightness_mean":
            np.nan,

        "contrast_std":
            np.nan,

        "blur_laplacian":
            np.nan,

        "underexposed_fraction":
            np.nan,

        "overexposed_fraction":
            np.nan,

        "specular_fraction":
            np.nan,
    }


def compute_image_qc(
    image_path,
    config,
):
    """
    Computes basic image-quality proxies for one
    capsule-endoscopy frame.

    These metrics are technical quality proxies.
    They are not clinical quality labels.
    """

    if pd.isna(image_path):
        return (
            build_unreadable_image_qc_record(
                image_path=image_path,
                error_message="missing_image_path",
            )
        )

    normalized_image_path = Path(
        str(image_path)
    )

    if not normalized_image_path.is_file():
        return (
            build_unreadable_image_qc_record(
                image_path=normalized_image_path,
                error_message="image_file_not_found",
            )
        )

    try:
        image = cv2.imread(
            str(normalized_image_path),
            cv2.IMREAD_COLOR,
        )

    except cv2.error as error:
        return (
            build_unreadable_image_qc_record(
                image_path=normalized_image_path,
                error_message=str(error),
            )
        )

    if image is None or image.size == 0:
        return (
            build_unreadable_image_qc_record(
                image_path=normalized_image_path,
                error_message="opencv_decode_failed",
            )
        )

    image_height, image_width = (
        image.shape[:2]
    )

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY,
    )

    hsv = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2HSV,
    )

    saturation = hsv[
        :,
        :,
        1,
    ]

    value = hsv[
        :,
        :,
        2,
    ]

    fov_mask = build_capsule_fov_mask(
        image_height=image_height,
        image_width=image_width,
        radius_fraction=config[
            "qc_fov_radius_fraction"
        ],
    )

    gray_pixels = gray[
        fov_mask
    ]

    saturation_pixels = saturation[
        fov_mask
    ]

    value_pixels = value[
        fov_mask
    ]

    if gray_pixels.size == 0:
        return (
            build_unreadable_image_qc_record(
                image_path=normalized_image_path,
                error_message="empty_fov_mask",
            )
        )

    laplacian = cv2.Laplacian(
        gray,
        cv2.CV_64F,
    )

    laplacian_pixels = laplacian[
        fov_mask
    ]

    return {
        "image_path":
            str(normalized_image_path),

        "image_readable":
            True,

        "image_read_error":
            pd.NA,

        "image_width":
            int(image_width),

        "image_height":
            int(image_height),

        "brightness_mean":
            float(
                gray_pixels.mean()
                / 255.0
            ),

        "contrast_std":
            float(
                gray_pixels.std()
                / 255.0
            ),

        # Higher values generally indicate a sharper image.
        "blur_laplacian":
            float(
                laplacian_pixels.var()
            ),

        "underexposed_fraction":
            float(
                np.mean(
                    gray_pixels
                    <
                    config[
                        "qc_underexposed_pixel_threshold"
                    ]
                )
            ),

        "overexposed_fraction":
            float(
                np.mean(
                    gray_pixels
                    >
                    config[
                        "qc_overexposed_pixel_threshold"
                    ]
                )
            ),

        "specular_fraction":
            float(
                np.mean(
                    (
                        value_pixels
                        >
                        config[
                            "qc_specular_value_threshold"
                        ]
                    )
                    &
                    (
                        saturation_pixels
                        <
                        config[
                            "qc_specular_saturation_threshold"
                        ]
                    )
                )
            ),
    }

### 33. Build image-QC manifest

In [ ]:
IMAGE_QC_FLAG_COLUMNS = [
    "qc_blur_outlier",
    "qc_contrast_outlier",
    "qc_low_brightness_outlier",
    "qc_high_brightness_outlier",
    "qc_requires_review",
]


IMAGE_QC_THRESHOLD_COLUMNS = [
    "blur_laplacian_lower_threshold",
    "contrast_std_lower_threshold",
    "brightness_mean_lower_threshold",
    "brightness_mean_upper_threshold",
]


IMAGE_QC_MANIFEST_COLUMNS = [
    "image_path",
    "used_by_labelled_manifest",
    "used_by_temporal_manifest",
    *[
        column
        for column in IMAGE_QC_COLUMNS
        if column != "image_path"
    ],
    *IMAGE_QC_FLAG_COLUMNS,
]


def validate_qc_quantiles(
    config,
):
    """
    Validates and returns dataset-level QC quantiles.
    """

    quantiles = pd.Series(
        {
            "blur":
                config[
                    "qc_blur_quantile"
                ],

            "contrast":
                config[
                    "qc_contrast_quantile"
                ],

            "brightness_low":
                config[
                    "qc_brightness_low_quantile"
                ],

            "brightness_high":
                config[
                    "qc_brightness_high_quantile"
                ],
        },
        dtype="float64",
    )

    invalid_quantile_mask = (
        quantiles.isna()
        | ~np.isfinite(quantiles)
        | quantiles.lt(0.0)
        | quantiles.gt(1.0)
    )

    if invalid_quantile_mask.any():
        raise ValueError(
            "QC quantiles must be finite values "
            "between 0 and 1."
        )

    if (
        quantiles["brightness_low"]
        >= quantiles["brightness_high"]
    ):
        raise ValueError(
            "The low brightness quantile must be "
            "smaller than the high brightness quantile."
        )

    return quantiles


def build_qc_image_inventory(
    labelled_manifest,
    temporal_manifest,
):
    """
    Builds one inventory row per unique physical image.

    The inventory records whether an image is used by the
    labelled-frame manifest, temporal manifest, or both.
    """

    required_labelled_columns = {
        "image_path",
    }

    required_temporal_columns = {
        "context_image_path",
    }

    missing_labelled_columns = sorted(
        required_labelled_columns
        - set(labelled_manifest.columns)
    )

    missing_temporal_columns = sorted(
        required_temporal_columns
        - set(temporal_manifest.columns)
    )

    if missing_labelled_columns:
        raise KeyError(
            "Labelled manifest is missing QC columns: "
            f"{missing_labelled_columns}"
        )

    if missing_temporal_columns:
        raise KeyError(
            "Temporal manifest is missing QC columns: "
            f"{missing_temporal_columns}"
        )

    labelled_images = (
        labelled_manifest[
            [
                "image_path",
            ]
        ]
        .rename(
            columns={
                "image_path":
                    "image_path"
            }
        )
        .assign(
            image_usage="labelled_manifest"
        )
    )

    temporal_images = (
        temporal_manifest[
            [
                "context_image_path",
            ]
        ]
        .rename(
            columns={
                "context_image_path":
                    "image_path"
            }
        )
        .assign(
            image_usage="temporal_manifest"
        )
    )

    image_usage = (
        pd.concat(
            [
                labelled_images,
                temporal_images,
            ],
            ignore_index=True,
        )
        .dropna(
            subset=[
                "image_path",
            ]
        )
        .assign(
            image_path=lambda data:
                data[
                    "image_path"
                ]
                .astype("string")
                .str.strip()
        )
        .loc[
            lambda data:
                data[
                    "image_path"
                ].ne("")
        ]
        .assign(
            used_by_labelled_manifest=lambda data:
                data[
                    "image_usage"
                ].eq(
                    "labelled_manifest"
                ),

            used_by_temporal_manifest=lambda data:
                data[
                    "image_usage"
                ].eq(
                    "temporal_manifest"
                ),
        )
    )

    return (
        image_usage
        .groupby(
            "image_path",
            as_index=False,
            sort=True,
            observed=True,
        )
        .agg(
            used_by_labelled_manifest=(
                "used_by_labelled_manifest",
                "max",
            ),

            used_by_temporal_manifest=(
                "used_by_temporal_manifest",
                "max",
            ),
        )
        .assign(
            used_by_labelled_manifest=lambda data:
                data[
                    "used_by_labelled_manifest"
                ].astype("boolean"),

            used_by_temporal_manifest=lambda data:
                data[
                    "used_by_temporal_manifest"
                ].astype("boolean"),
        )
        .reset_index(
            drop=True
        )
    )


def compute_qc_for_image_inventory(
    image_inventory,
    config,
):
    """
    Computes QC metrics once for every unique image.
    """

    if image_inventory.empty:
        raise ValueError(
            "Cannot compute QC for an empty image inventory."
        )

    qc_record_series = (
        image_inventory[
            "image_path"
        ]
        .map(
            lambda image_path:
                compute_image_qc(
                    image_path=image_path,
                    config=config,
                )
        )
    )

    qc_metrics = (
        pd.DataFrame.from_records(
            qc_record_series.tolist(),
            columns=IMAGE_QC_COLUMNS,
        )
    )

    if not qc_metrics[
        "image_path"
    ].is_unique:
        raise RuntimeError(
            "QC computation produced duplicate image paths."
        )

    return (
        image_inventory
        .merge(
            qc_metrics,
            on="image_path",
            how="left",
            validate="one_to_one",
        )
        .assign(
            image_readable=lambda data:
                data[
                    "image_readable"
                ].astype("boolean"),

            image_read_error=lambda data:
                data[
                    "image_read_error"
                ].astype("string"),
        )
    )


def derive_image_qc_thresholds(
    image_qc_manifest,
    config,
):
    """
    Derives technical outlier thresholds from readable
    images using the configured quantiles.
    """

    quantiles = validate_qc_quantiles(
        config=config
    )

    readable_mask = (
        image_qc_manifest[
            "image_readable"
        ]
        .fillna(False)
        .astype(bool)
    )

    readable_metrics = (
        image_qc_manifest.loc[
            readable_mask,
            [
                "brightness_mean",
                "contrast_std",
                "blur_laplacian",
            ],
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    invalid_metric_mask = (
        readable_metrics.isna()
        .any(axis=1)
        |
        ~np.isfinite(
            readable_metrics
        )
        .all(axis=1)
    )

    if invalid_metric_mask.any():
        invalid_paths = (
            image_qc_manifest.loc[
                readable_metrics.index[
                    invalid_metric_mask
                ],
                "image_path",
            ]
            .tolist()
        )

        raise RuntimeError(
            "Readable images produced invalid QC metrics: "
            f"{invalid_paths}"
        )

    if readable_metrics.empty:
        raise RuntimeError(
            "No readable images are available for "
            "deriving QC thresholds."
        )

    return pd.DataFrame.from_records(
        [
            {
                "blur_laplacian_lower_threshold":
                    readable_metrics[
                        "blur_laplacian"
                    ].quantile(
                        quantiles["blur"]
                    ),

                "contrast_std_lower_threshold":
                    readable_metrics[
                        "contrast_std"
                    ].quantile(
                        quantiles["contrast"]
                    ),

                "brightness_mean_lower_threshold":
                    readable_metrics[
                        "brightness_mean"
                    ].quantile(
                        quantiles[
                            "brightness_low"
                        ]
                    ),

                "brightness_mean_upper_threshold":
                    readable_metrics[
                        "brightness_mean"
                    ].quantile(
                        quantiles[
                            "brightness_high"
                        ]
                    ),
            }
        ],
        columns=IMAGE_QC_THRESHOLD_COLUMNS,
    )


def add_image_qc_flags(
    image_qc_manifest,
    image_qc_thresholds,
):
    """
    Adds technical review flags without interpreting
    them as clinical-quality labels.
    """

    if len(image_qc_thresholds) != 1:
        raise ValueError(
            "Image QC thresholds must contain exactly "
            "one row."
        )

    thresholds = (
        image_qc_thresholds.iloc[0]
    )

    readable_mask = (
        image_qc_manifest[
            "image_readable"
        ]
        .fillna(False)
        .astype(bool)
    )

    result = (
        image_qc_manifest
        .assign(
            qc_blur_outlier=(
                readable_mask
                &
                image_qc_manifest[
                    "blur_laplacian"
                ].le(
                    thresholds[
                        "blur_laplacian_lower_threshold"
                    ]
                )
            ),

            qc_contrast_outlier=(
                readable_mask
                &
                image_qc_manifest[
                    "contrast_std"
                ].le(
                    thresholds[
                        "contrast_std_lower_threshold"
                    ]
                )
            ),

            qc_low_brightness_outlier=(
                readable_mask
                &
                image_qc_manifest[
                    "brightness_mean"
                ].le(
                    thresholds[
                        "brightness_mean_lower_threshold"
                    ]
                )
            ),

            qc_high_brightness_outlier=(
                readable_mask
                &
                image_qc_manifest[
                    "brightness_mean"
                ].ge(
                    thresholds[
                        "brightness_mean_upper_threshold"
                    ]
                )
            ),
        )
        .assign(
            qc_requires_review=lambda data:
                (
                    ~readable_mask
                    | data[
                        "qc_blur_outlier"
                    ]
                    | data[
                        "qc_contrast_outlier"
                    ]
                    | data[
                        "qc_low_brightness_outlier"
                    ]
                    | data[
                        "qc_high_brightness_outlier"
                    ]
                )
        )
        .assign(
            **{
                column: lambda data, column=column:
                    data[
                        column
                    ].astype("boolean")

                for column in (
                    IMAGE_QC_FLAG_COLUMNS
                )
            }
        )
        .loc[
            :,
            IMAGE_QC_MANIFEST_COLUMNS,
        ]
        .sort_values(
            "image_path",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return result


if CONFIG[
    "qc_enabled"
]:

    image_qc_inventory = (
        build_qc_image_inventory(
            labelled_manifest=(
                labelled_frame_manifest
            ),
            temporal_manifest=(
                temporal_manifest
            ),
        )
    )

    image_qc_manifest = (
        image_qc_inventory
        .pipe(
            compute_qc_for_image_inventory,
            config=CONFIG,
        )
    )

    image_qc_thresholds = (
        derive_image_qc_thresholds(
            image_qc_manifest=(
                image_qc_manifest
            ),
            config=CONFIG,
        )
    )

    image_qc_manifest = (
        image_qc_manifest
        .pipe(
            add_image_qc_flags,
            image_qc_thresholds=(
                image_qc_thresholds
            ),
        )
    )

else:

    image_qc_inventory = pd.DataFrame(
        columns=[
            "image_path",
            "used_by_labelled_manifest",
            "used_by_temporal_manifest",
        ]
    )

    image_qc_thresholds = pd.DataFrame(
        columns=IMAGE_QC_THRESHOLD_COLUMNS
    )

    image_qc_manifest = pd.DataFrame(
        columns=IMAGE_QC_MANIFEST_COLUMNS
    )


image_qc_summary = (
    pd.DataFrame.from_records(
        [
            {
                "total_unique_images":
                    len(
                        image_qc_manifest
                    ),

                "readable_images":
                    int(
                        image_qc_manifest[
                            "image_readable"
                        ]
                        .fillna(False)
                        .sum()
                    ),

                "review_required_images":
                    int(
                        image_qc_manifest[
                            "qc_requires_review"
                        ]
                        .fillna(False)
                        .sum()
                    ),
            }
        ]
    )
)


display(
    image_qc_summary
)

display(
    image_qc_thresholds
)

display(
    image_qc_manifest.head()
)

### 34. Persist the complete Phase 2 data lake

In [ ]:
DATA_LAKE_TABLE_SPECS = pd.DataFrame.from_records(
    [
        # ----------------------------------------------------------
        # Core manifests
        # ----------------------------------------------------------

        {
            "artifact_name":
                "raw_metadata_snapshot",

            "variable_name":
                "df_raw",

            "artifact_group":
                "manifest",

            "filename":
                "raw_metadata_snapshot.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "normalized_annotation_metadata",

            "variable_name":
                "df",

            "artifact_group":
                "manifest",

            "filename":
                "normalized_annotation_metadata.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "video_manifest",

            "variable_name":
                "video_manifest",

            "artifact_group":
                "manifest",

            "filename":
                "video_manifest.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "video_split_manifest",

            "variable_name":
                "video_split_manifest",

            "artifact_group":
                "manifest",

            "filename":
                "video_split_manifest.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "labelled_frame_manifest",

            "variable_name":
                "labelled_frame_manifest",

            "artifact_group":
                "manifest",

            "filename":
                "labelled_frame_manifest.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "video_class_matrix",

            "variable_name":
                "video_class_matrix",

            "artifact_group":
                "manifest",

            "filename":
                "video_class_matrix.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "domain_source_videos",

            "variable_name":
                "domain_source_videos",

            "artifact_group":
                "manifest",

            "filename":
                "domain_source_videos.parquet",

            "required":
                bool(
                    CONFIG[
                        "domain_adaptation_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "domain_adaptation_manifest",

            "variable_name":
                "domain_adaptation_manifest",

            "artifact_group":
                "manifest",

            "filename":
                "domain_adaptation_manifest.parquet",

            "required":
                bool(
                    CONFIG[
                        "domain_adaptation_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "finding_segment_manifest",

            "variable_name":
                "finding_segment_manifest",

            "artifact_group":
                "manifest",

            "filename":
                "finding_segment_manifest.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "temporal_extraction_requests",

            "variable_name":
                "temporal_extraction_requests",

            "artifact_group":
                "manifest",

            "filename":
                "temporal_extraction_requests.parquet",

            "required":
                bool(
                    CONFIG[
                        "temporal_context_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "temporal_manifest",

            "variable_name":
                "temporal_manifest",

            "artifact_group":
                "manifest",

            "filename":
                "temporal_manifest.parquet",

            "required":
                bool(
                    CONFIG[
                        "temporal_context_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "image_qc_inventory",

            "variable_name":
                "image_qc_inventory",

            "artifact_group":
                "manifest",

            "filename":
                "image_qc_inventory.parquet",

            "required":
                bool(
                    CONFIG[
                        "qc_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "image_qc_manifest",

            "variable_name":
                "image_qc_manifest",

            "artifact_group":
                "manifest",

            "filename":
                "image_qc_manifest.parquet",

            "required":
                bool(
                    CONFIG[
                        "qc_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "image_qc_thresholds",

            "variable_name":
                "image_qc_thresholds",

            "artifact_group":
                "manifest",

            "filename":
                "image_qc_thresholds.parquet",

            "required":
                bool(
                    CONFIG[
                        "qc_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "raw_dataset_inventory_report",

            "variable_name":
                "raw_dataset_inventory_report",

            "artifact_group":
                "report",

            "filename":
                "raw_dataset_inventory_report.parquet",

            "required":
                True,
        },

        # ----------------------------------------------------------
        # Audit and validation reports
        # ----------------------------------------------------------

        {
            "artifact_name":
                "video_inventory_summary",

            "variable_name":
                "video_inventory_summary",

            "artifact_group":
                "report",

            "filename":
                "video_inventory_summary.parquet",

            "required":
                False,
        },

        {
            "artifact_name":
                "video_counts",

            "variable_name":
                "video_counts",

            "artifact_group":
                "report",

            "filename":
                "video_counts.parquet",

            "required":
                False,
        },

        {
            "artifact_name":
                "clinical_taxonomy_report",

            "variable_name":
                "clinical_taxonomy_report",

            "artifact_group":
                "report",

            "filename":
                "clinical_taxonomy_report.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "frame_alignment_report",

            "variable_name":
                "frame_alignment_report",

            "artifact_group":
                "report",

            "filename":
                "frame_alignment_report.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "video_class_support",

            "variable_name":
                "video_class_support",

            "artifact_group":
                "report",

            "filename":
                "video_class_support.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "video_role_report",

            "variable_name":
                "video_role_report",

            "artifact_group":
                "report",

            "filename":
                "video_role_report.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "domain_adaptation_summary",

            "variable_name":
                "domain_adaptation_summary",

            "artifact_group":
                "report",

            "filename":
                "domain_adaptation_summary.parquet",

            "required":
                bool(
                    CONFIG[
                        "domain_adaptation_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "temporal_window_report",

            "variable_name":
                "temporal_window_report",

            "artifact_group":
                "report",

            "filename":
                "temporal_window_report.parquet",

            "required":
                bool(
                    CONFIG[
                        "temporal_context_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "temporal_extraction_report",

            "variable_name":
                "extracted_temporal_frames",

            "artifact_group":
                "report",

            "filename":
                "temporal_extraction_report.parquet",

            "required":
                bool(
                    CONFIG[
                        "temporal_context_enabled"
                    ]
                    and
                    CONFIG[
                        "temporal_context_extract_frames"
                    ]
                ),
        },

        {
            "artifact_name":
                "dataset_characteristics_report",

            "variable_name":
                "dataset_characteristics_report",

            "artifact_group":
                "report",

            "filename":
                "dataset_characteristics_report.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "image_qc_summary",

            "variable_name":
                "image_qc_summary",

            "artifact_group":
                "report",

            "filename":
                "image_qc_summary.parquet",

            "required":
                bool(
                    CONFIG[
                        "qc_enabled"
                    ]
                ),
        },

        {
            "artifact_name":
                "labelled_frame_bounds_report",

            "variable_name":
                "labelled_frame_bounds_report",

            "artifact_group":
                "report",

            "filename":
                "labelled_frame_bounds_report.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "frame_inventory_report",

            "variable_name":
                "frame_inventory_report",

            "artifact_group":
                "report",

            "filename":
                "frame_inventory_report.parquet",

            "required":
                True,
        },

        {
            "artifact_name":
                "temporal_extraction_failures",

            "variable_name":
                "temporal_extraction_failures",

            "artifact_group":
                "report",

            "filename":
                "temporal_extraction_failures.parquet",

            "required":
                bool(
                    CONFIG["temporal_context_enabled"]
                    and
                    CONFIG["temporal_context_extract_frames"]
                    ),
        },

        {
            "artifact_name":
                "split_class_distribution_report",

            "variable_name":
                "split_class_distribution_report",

            "artifact_group":
                "report",

            "filename":
                "split_class_distribution_report.parquet",

            "required":
                True,
        },
    ]
)


def build_video_split_manifest(
    video_manifest,
):
    """
    Builds the persistent video-level split and role table.
    """

    required_columns = {
        "video_key",
        "video_annotation_type",
        "supervised_split",
        "data_role",
    }

    missing_columns = sorted(
        required_columns
        - set(video_manifest.columns)
    )

    if missing_columns:
        raise KeyError(
            "Cannot persist video splits. Missing columns: "
            f"{missing_columns}"
        )

    return (
        video_manifest.loc[
            :,
            [
                "video_key",
                "video_annotation_type",
                "supervised_split",
                "data_role",
            ],
        ]
        .sort_values(
            "video_key",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


def prepare_dataframe_for_parquet(
    dataframe,
):
    """
    Produces a storage-safe DataFrame while preserving
    meaningful named indexes.
    """

    if not isinstance(
        dataframe,
        pd.DataFrame,
    ):
        raise TypeError(
            "Only pandas DataFrames can be persisted "
            "as data-lake tables."
        )

    result = dataframe.copy()

    has_meaningful_index = (
        not isinstance(
            result.index,
            pd.RangeIndex,
        )
        or result.index.name is not None
    )

    if has_meaningful_index:
        result = result.reset_index()
    else:
        result = result.reset_index(
            drop=True
        )

    path_columns = [
        column
        for column in result.columns
        if (
            str(column).endswith("_path")
            or str(column).endswith("_dir")
        )
    ]

    if path_columns:
        result = result.assign(
            **{
                column:
                    result[
                        column
                    ].map(
                        lambda value:
                            (
                                str(value)
                                if pd.notna(value)
                                else pd.NA
                            )
                    )
                    .astype("string")

                for column in path_columns
            }
        )

    return result


def write_parquet_atomically(
    dataframe,
    output_path,
):
    """
    Writes a Parquet table through a temporary file so
    an interrupted run does not replace a valid artifact
    with a partially written file.
    """

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        /
        (
            f".{output_path.stem}."
            f"{CONFIG['run_id']}.tmp.parquet"
        )
    )

    dataframe.to_parquet(
        temporary_path,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    temporary_path.replace(
        output_path
    )

    return output_path


def json_default(
    value,
):
    """
    Converts pipeline objects into JSON-safe values.
    """

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(
        value,
        (
            set,
            frozenset,
        ),
    ):
        return sorted(
            value
        )

    raise TypeError(
        f"Object is not JSON serializable: {type(value)}"
    )


def write_json_atomically(
    payload,
    output_path,
):
    """
    Writes a JSON artifact atomically.
    """

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        /
        (
            f".{output_path.stem}."
            f"{CONFIG['run_id']}.tmp.json"
        )
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as output_file:
        json.dump(
            payload,
            output_file,
            indent=2,
            sort_keys=True,
            default=json_default,
        )

    temporary_path.replace(
        output_path
    )

    return output_path


DATA_LAKE_READBACK_REPORT_COLUMNS = [
    "artifact_name",
    "file_path",
    "expected_row_count",
    "actual_row_count",
    "expected_column_count",
    "actual_column_count",
    "file_exists",
    "read_success",
    "row_count_matches",
    "column_count_matches",
    "validation_passed",
    "validation_error",
]


def build_data_lake_readback_report(
    data_lake_catalog,
):
    """
    Reopens every persisted Parquet table and compares
    its physical dimensions with the artifact catalog.

    The persisted files are read but not modified.
    """

    required_columns = {
        "artifact_name",
        "file_path",
        "row_count",
        "column_count",
    }

    missing_columns = sorted(
        required_columns
        - set(data_lake_catalog.columns)
    )

    if missing_columns:
        raise KeyError(
            "Data Lake catalog is missing read-back "
            f"columns: {missing_columns}"
        )

    if data_lake_catalog.empty:
        raise ValueError(
            "Cannot validate an empty Data Lake catalog."
        )

    if data_lake_catalog[
        "artifact_name"
    ].duplicated().any():
        raise ValueError(
            "Data Lake catalog contains duplicate "
            "artifact names."
        )

    if data_lake_catalog[
        "file_path"
    ].duplicated().any():
        raise ValueError(
            "Data Lake catalog contains duplicate "
            "physical file paths."
        )

    validation_records = []

    for artifact in data_lake_catalog.itertuples(
        index=False
    ):
        artifact_path = Path(
            artifact.file_path
        )

        expected_row_count = int(
            artifact.row_count
        )

        expected_column_count = int(
            artifact.column_count
        )

        file_exists = (
            artifact_path.is_file()
        )

        read_success = False
        actual_row_count = pd.NA
        actual_column_count = pd.NA
        validation_error = pd.NA

        if not file_exists:
            validation_error = (
                "persisted_file_not_found"
            )

        else:
            try:
                persisted_table = pd.read_parquet(
                    artifact_path,
                    engine="pyarrow",
                )

                actual_row_count = len(
                    persisted_table
                )

                actual_column_count = len(
                    persisted_table.columns
                )

                read_success = True

            except Exception as error:
                validation_error = (
                    f"{type(error).__name__}: "
                    f"{error}"
                )

        row_count_matches = bool(
            read_success
            and actual_row_count
            == expected_row_count
        )

        column_count_matches = bool(
            read_success
            and actual_column_count
            == expected_column_count
        )

        validation_passed = bool(
            file_exists
            and read_success
            and row_count_matches
            and column_count_matches
        )

        validation_records.append(
            {
                "artifact_name":
                    artifact.artifact_name,

                "file_path":
                    str(artifact_path),

                "expected_row_count":
                    expected_row_count,

                "actual_row_count":
                    actual_row_count,

                "expected_column_count":
                    expected_column_count,

                "actual_column_count":
                    actual_column_count,

                "file_exists":
                    file_exists,

                "read_success":
                    read_success,

                "row_count_matches":
                    row_count_matches,

                "column_count_matches":
                    column_count_matches,

                "validation_passed":
                    validation_passed,

                "validation_error":
                    validation_error,
            }
        )

    boolean_columns = [
        "file_exists",
        "read_success",
        "row_count_matches",
        "column_count_matches",
        "validation_passed",
    ]

    integer_columns = [
        "expected_row_count",
        "actual_row_count",
        "expected_column_count",
        "actual_column_count",
    ]

    return (
        pd.DataFrame.from_records(
            validation_records,
            columns=(
                DATA_LAKE_READBACK_REPORT_COLUMNS
            ),
        )
        .assign(
            **{
                column:
                    lambda data, column=column:
                        data[
                            column
                        ].astype("boolean")

                for column in boolean_columns
            },

            **{
                column:
                    lambda data, column=column:
                        pd.to_numeric(
                            data[
                                column
                            ],
                            errors="coerce",
                        )
                        .astype("Int64")

                for column in integer_columns
            },
        )
        .sort_values(
            "artifact_name",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


def validate_data_lake_readback_report(
    readback_report,
):
    """
    Stops the pipeline when a persisted Parquet artifact
    cannot be reopened or has unexpected dimensions.
    """

    failed_artifacts = (
        readback_report.loc[
            ~readback_report[
                "validation_passed"
            ]
        ]
    )

    if not failed_artifacts.empty:
        raise RuntimeError(
            "Data Lake read-back validation failed: "
            f"{failed_artifacts.to_dict(orient='records')}"
        )

    return readback_report

def persist_data_lake_tables(
    table_specs,
    runtime_namespace,
    directories,
):
    """
    Persists all declared Data Lake tables and returns
    one catalog row per saved artifact.
    """

    missing_required_artifacts = (
        table_specs.loc[
            table_specs[
                "required"
            ]
            &
            ~table_specs[
                "variable_name"
            ].isin(
                runtime_namespace
            ),
            "variable_name",
        ]
        .tolist()
    )

    if missing_required_artifacts:
        raise NameError(
            "Required Data Lake artifacts were not built: "
            f"{missing_required_artifacts}"
        )

    catalog_records = []

    for spec in table_specs.itertuples(
        index=False
    ):
        if (
            spec.variable_name
            not in runtime_namespace
        ):
            continue

        source_table = (
            runtime_namespace[
                spec.variable_name
            ]
        )

        if not isinstance(
            source_table,
            pd.DataFrame,
        ):
            if spec.required:
                raise TypeError(
                    "Required artifact is not a DataFrame: "
                    f"{spec.variable_name}"
                )

            continue

        storage_table = (
            prepare_dataframe_for_parquet(
                dataframe=source_table
            )
        )

        output_directory = (
            directories[
                spec.artifact_group
            ]
        )

        output_path = (
            output_directory
            / spec.filename
        )

        saved_path = (
            write_parquet_atomically(
                dataframe=storage_table,
                output_path=output_path,
            )
        )

        catalog_records.append(
            {
                "artifact_name":
                    spec.artifact_name,

                "source_variable":
                    spec.variable_name,

                "artifact_group":
                    spec.artifact_group,

                "file_path":
                    str(
                        saved_path
                    ),

                "file_relpath":
                    path_relative_to_storage(
                        value=saved_path,
                        storage_root=(
                            CONFIG[
                                "storage_root"
                            ]
                        ),
                    ),

                "row_count":
                    len(
                        storage_table
                    ),

                "column_count":
                    len(
                        storage_table.columns
                    ),

                "file_size_bytes":
                    saved_path.stat().st_size,

                "run_id":
                    CONFIG[
                        "run_id"
                    ],
            }
        )

    return (
        pd.DataFrame.from_records(
            catalog_records
        )
        .sort_values(
            [
                "artifact_group",
                "artifact_name",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------------
# Build the explicit video-level split artifact
# ------------------------------------------------------------------

video_split_manifest = (
    build_video_split_manifest(
        video_manifest=video_manifest
    )
)


# ------------------------------------------------------------------
# Persistent Google Drive destinations
# ------------------------------------------------------------------

data_lake_directories = {
    "manifest":
        Path(
            DIRS[
                "manifests_dir"
            ]
        ),

    "report":
        Path(
            DIRS[
                "reports_dir"
            ]
        ),
}


for directory in (
    data_lake_directories.values()
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------------
# Persist every declared table
# ------------------------------------------------------------------

data_lake_catalog = (
    persist_data_lake_tables(
        table_specs=(
            DATA_LAKE_TABLE_SPECS
        ),
        runtime_namespace=globals(),
        directories=(
            data_lake_directories
        ),
    )
)

# ------------------------------------------------------------------
# Read back and validate persisted Parquet tables
# ------------------------------------------------------------------

data_lake_readback_report = (
    build_data_lake_readback_report(
        data_lake_catalog=(
            data_lake_catalog
        )
    )
)


data_lake_readback_report_path = (
    Path(
        DIRS[
            "reports_dir"
        ]
    )
    / "data_lake_readback_report.parquet"
)


write_parquet_atomically(
    dataframe=(
        data_lake_readback_report
    ),
    output_path=(
        data_lake_readback_report_path
    ),
)


readback_catalog_record = (
    pd.DataFrame.from_records(
        [
            {
                "artifact_name":
                    "data_lake_readback_report",

                "source_variable":
                    "data_lake_readback_report",

                "artifact_group":
                    "report",

                "file_path":
                    str(
                        data_lake_readback_report_path
                    ),

                "file_relpath":
                    path_relative_to_storage(
                        value=(
                            data_lake_readback_report_path
                        ),
                        storage_root=(
                            CONFIG[
                                "storage_root"
                            ]
                        ),
                    ),

                "row_count":
                    len(
                        data_lake_readback_report
                    ),

                "column_count":
                    len(
                        data_lake_readback_report.columns
                    ),

                "file_size_bytes":
                    (
                        data_lake_readback_report_path
                        .stat()
                        .st_size
                    ),

                "run_id":
                    CONFIG[
                        "run_id"
                    ],
            }
        ]
    )
)


data_lake_catalog = (
    pd.concat(
        [
            data_lake_catalog,
            readback_catalog_record,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "artifact_group",
            "artifact_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------------
# Persist run configuration and derived parameters
# ------------------------------------------------------------------

data_lake_configuration = {
    "run_id":
        CONFIG[
            "run_id"
        ],

    "saved_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "config":
        CONFIG,

    "directories": {
        name:
            str(path)

        for name, path
        in DIRS.items()
    },

    "derived_parameters": {
        "frame_index_offset":
            int(
                FRAME_INDEX_OFFSET
            ),

        "bbox_spec":
            BBOX_SPEC,
    },
}


configuration_path = (
    Path(
        DIRS[
            "configs_dir"
        ]
    )
    / "phase2_data_lake_configuration.json"
)


write_json_atomically(
    payload=(
        data_lake_configuration
    ),
    output_path=(
        configuration_path
    ),
)


# ------------------------------------------------------------------
# Persist the artifact catalog last
# ------------------------------------------------------------------

data_lake_catalog_path = (
    Path(
        DIRS[
            "manifests_dir"
        ]
    )
    / "phase2_data_lake_catalog.parquet"
)


write_parquet_atomically(
    dataframe=(
        data_lake_catalog
    ),
    output_path=(
        data_lake_catalog_path
    ),
)

data_lake_readback_report = (
    data_lake_readback_report
    .pipe(
        validate_data_lake_readback_report
    )
)


print(
    "PASS: persisted Parquet tables were reopened "
    "and validated."
)


print(
    "Phase 2 Data Lake saved successfully."
)

print(
    "Persistent storage root:",
    CONFIG[
        "storage_root"
    ],
)

print(
    "Saved tables:",
    f"{len(data_lake_catalog):,}",
)

print(
    "Artifact catalog:",
    data_lake_catalog_path,
)

print(
    "Configuration snapshot:",
    configuration_path,
)


display(
    data_lake_catalog[
        [
            "artifact_name",
            "artifact_group",
            "row_count",
            "file_relpath",
        ]
    ]
)